In [709]:
import pathlib as Path
import numpy as np
import pandas as pd


## Create dataframe from embeddings

In [710]:
#-----Frequency Embeddings dataframe creation-----
def create_embeddings_dataframe(root_path):
    emb_root = Path.Path(root_path)
    emb_files = emb_root.rglob("embeddings_samples.npz")
    
    all_data = []
    
    for emb_file in emb_files:
        emb = np.load(emb_file)
        X = emb['X']
        y = emb['y']
        subjects = emb['subs']
        
        for i in range(X.shape[0]):
            data_point = {
                'embedding': X[i],
                'label': y[i],
                'subject': subjects[i],
                'file_path': str(emb_file)
            }
            all_data.append(data_point)
    
    df = pd.DataFrame(all_data)
    return df


## Clean embedding DF

In [711]:

def df_cleaning(df, emb_size=384):
        # --- Expand embedding column into 381 separate columns ---
    embedding_df = pd.DataFrame(df["embedding"].tolist(),
                                columns=[f"emb_{i}" for i in range(emb_size)])

    # --- Concatenate back to the original DataFrame (optional) ---
    df_expanded = pd.concat([df.drop(columns=["embedding"]), embedding_df], axis=1)
    return df_expanded

In [712]:
#-----Frequency Embeddings dataframe creation-----
freq_emb= create_embeddings_dataframe("C://dev//dolphin_initial_testing//DOLPHIN//out_prelbd_task_wise//frequency_emb_stored//")
#-----Temporal Embeddings dataframe creation-----
temp_emb= create_embeddings_dataframe("C://dev//dolphin_initial_testing//DOLPHIN//out_prelbd_task_wise//temporal_emb_stored//")
#-----Combined Embeddings dataframe creation-----
comb_emb= create_embeddings_dataframe("C://dev//dolphin_initial_testing//DOLPHIN//out_prelbd_task_wise//emb_stored//")

freq_emb_df= df_cleaning(freq_emb, emb_size=384)
temp_emb_df= df_cleaning(temp_emb, emb_size=384)
comb_emb_df= df_cleaning(comb_emb, emb_size=768)

In [713]:

metadata_path = ('C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_metadata//preDLB_shared(PSY_RAW).csv')
df_metadata = pd.read_csv(metadata_path, sep=";", encoding="utf-8-sig")

In [714]:
clinical_data_path = ('C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_metadata//clinical_data_csv.csv')
df_clinical = pd.read_csv(clinical_data_path, sep=",", encoding="utf-8-sig")

In [715]:
handcrafted_features_path = ('C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002_hf//corpus_LBD_CZ_002_writing_results_table_original_filtered_extended.csv')
df_handcrafted_features = pd.read_csv(handcrafted_features_path, sep=";", encoding="utf-8-sig")

In [716]:
import re
import pandas as pd

def append_col_when_main_contains_source(
    df_main, df_source, *, 
    match_col_main="subject",            # in df_main
    match_col_source="ID_1.meranie",     # in df_source
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
):
    df_main = df_main.copy()
    df_source = df_source.copy()

    # normalize + keep only rows in source with non-missing values
    df_main[match_col_main] = df_main[match_col_main].astype(str).str.strip()
    df_source[match_col_source] = df_source[match_col_source].astype(str).str.strip()
    df_source = df_source.dropna(subset=[value_col])

    # init target column
    if new_col_name not in df_main:
        df_main[new_col_name] = pd.NA

    # for each source row, mark all main rows whose subject CONTAINS the source token
    for _, r in df_source.iterrows():
        token = r[match_col_source]
        if not token:
            continue
        mask = df_main[match_col_main].str.contains(re.escape(token), na=False, case=not case)
        # write only where we don't have a value yet (keeps first hit)
        to_set = mask & df_main[new_col_name].isna()
        df_main.loc[to_set, new_col_name] = r[value_col]

    return df_main

In [717]:
freq_emb_df_lbl = append_col_when_main_contains_source(
    df_main=freq_emb_df,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)

In [443]:
freq_emb_df_lbl.to_csv("LBD_CZ_002_COBEN_dfs/freq_emb_df_half_lbl.csv", sep=";")

In [718]:

temp_emb_df_lbl = append_col_when_main_contains_source(
    df_main=temp_emb_df,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)

In [445]:

temp_emb_df_lbl.to_csv("LBD_CZ_002_COBEN_dfs/temp_emb_df_half__lbl.csv", sep=";")

In [719]:

comb_emb_df_lbl = append_col_when_main_contains_source(
    df_main=comb_emb_df,
    df_source=df_metadata,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)

In [447]:
comb_emb_df_lbl.to_csv("LBD_CZ_002_COBEN_dfs/comb_emb_df_half__lbl.csv", sep=";")

In [720]:

df_handcrafted_lbl = append_col_when_main_contains_source(
    df_main=df_handcrafted_features,
    df_source=df_metadata,
    match_col_main="ID",
    match_col_source="ID_1.meranie",
    value_col="HC0_nHC1_MCI2_MCILB3_baseline",
    new_col_name="diagnosis",
    case=False
)


In [449]:

df_handcrafted_lbl.to_csv("LBD_CZ_002_COBEN_dfs/df_handcrafted_half_lbl.csv", sep=";")

In [721]:
#temp_emb_df_complet_lbl = pd.read_csv("LBD_CZ_002_COBEN_dfs/temp_emb_df_lbl.csv", sep=";")

#freq_emb_df_complet_lbl= pd.read_csv("./LBD_CZ_002_COBEN_dfs/freq_emb_df_lbl.csv", sep=";")

#comb_emb_df_complet_lbl= pd.read_csv("LBD_CZ_002_COBEN_dfs/comb_emb_df_lbl.csv", sep=";")

hf_emb_df_complet_lbl= pd.read_csv("LBD_CZ_002_COBEN_dfs/df_handcrafted_lbl.csv", sep=";")


In [722]:

import pandas as pd

def assign_diagnosis(df: pd.DataFrame,
                     subject_col: str = "subject",
                     diagnosis_col: str = "diagnosis") -> pd.DataFrame:
    """Populate `diagnosis_col` based on text in `subject_col`.

    - rows whose subject contains '#' are left untouched
    - 'HC' → 0.0, 'AD' → 2.0, 'PD' → 4.0
    """
    if diagnosis_col not in df.columns:
        df[diagnosis_col] = pd.NA

    subj = df[subject_col].astype(str)
    has_hash = subj.str.contains("#", na=False)

    mapping = {"HC": 0.0, "AD": 2.0, "PD": 4.0}
    for key, value in mapping.items():
        mask = (~has_hash) & subj.str.contains(key, na=False)
        df.loc[mask, diagnosis_col] = value

    return df


In [723]:
temp_emb_df_complet_lbl = assign_diagnosis(temp_emb_df_lbl)

freq_emb_df_complet_lbl= assign_diagnosis(freq_emb_df_lbl)

comb_emb_df_complet_lbl=assign_diagnosis(comb_emb_df_lbl)


In [724]:
import pandas as pd

def assign_diagnosis(df: pd.DataFrame,
                     subject_col: str = "subject",
                     diagnosis_col: str = "diagnosis") -> pd.DataFrame:
    """Populate `diagnosis_col` based on text in `subject_col`.

    - rows whose subject contains '#' are left untouched
    - 'HC' → 0.0, 'AD' → 2.0, 'PD' → 4.0
    """
    if diagnosis_col not in df.columns:
        df[diagnosis_col] = pd.NA

    subj = df[subject_col].astype(str)
    has_hash = subj.str.contains("#", na=False)

    mapping = {"HC": 0.0, "AD": 2.0, "PD": 4.0}
    for key, value in mapping.items():
        mask = (~has_hash) & subj.str.contains(key, na=False)
        df.loc[mask, diagnosis_col] = value

    return df


In [725]:
def append_multiple_cols_when_main_contains_source(
    df_main, df_source, *,
    match_col_main="subject",
    match_col_source="ID_1.meranie",
    value_cols=("HC0_nHC1_MCI2_MCILB3_baseline",),
    case=False
):
    df_main = df_main.copy()
    df_source = df_source.copy()

    # normalize + clean
    df_main[match_col_main] = df_main[match_col_main].astype(str).str.strip()
    df_source[match_col_source] = df_source[match_col_source].astype(str).str.strip()

    # initialize missing columns
    for col in value_cols:
        if col not in df_main.columns:
            df_main[col] = pd.NA

    # iterate through source
    for _, r in df_source.iterrows():
        token = r[match_col_source]
        if not token or pd.isna(token):
            continue

        # use 'case' argument as passed (was inverted before)
        mask = df_main[match_col_main].str.contains(re.escape(str(token)), na=False, case=case)
        to_set = mask

        # assign all defined value columns
        for col in value_cols:
            if col not in r or pd.isna(r[col]):
                continue
            df_main.loc[to_set & df_main[col].isna(), col] = r[col]

    return df_main


In [726]:
clinical_hf_emb_df_complet_lbl = append_multiple_cols_when_main_contains_source(
    df_main=hf_emb_df_complet_lbl,
    df_source=df_clinical,
    match_col_main="ID",
    match_col_source="#personalID",
    value_cols=["age",
                "gender",
                "MOCA",
                "education type",
                "education length",
                "memory z-score",
                "visuo-spatial z-score",
                "attention z-score",
                "executive function z-score",
                "GDS"]
)

In [727]:
temp_emb_df_complet_lbl_2 = temp_emb_df_complet_lbl.drop(temp_emb_df_complet_lbl.columns[0], axis=1)
freq_emb_df_complet_lbl_2 = freq_emb_df_complet_lbl.drop(freq_emb_df_complet_lbl.columns[0], axis=1)
comb_emb_df_complet_lbl_2 = comb_emb_df_complet_lbl.drop(comb_emb_df_complet_lbl.columns[0], axis=1)
hf_emb_df_complet_lbl_2 = hf_emb_df_complet_lbl.drop(hf_emb_df_complet_lbl.columns[0], axis=1)

In [728]:

hf_emb_df_complet_lbl = hf_emb_df_complet_lbl.drop(hf_emb_df_complet_lbl.columns[0], axis=1)

# Spearsman correlation / finding covariants

In [729]:

cols_to_convert = clinical_hf_emb_df_complet_lbl.columns[1:]

clinical_hf_emb_df_complet_lbl[cols_to_convert] = (
    clinical_hf_emb_df_complet_lbl[cols_to_convert]
    .apply(lambda col: pd.to_numeric(col.astype(str).str.replace(',', '.'), errors='coerce'))
)

In [461]:
clinical_hf_emb_df_complet_lbl.dtypes

Unnamed: 0                                                    int64
ID                                                          float64
w.cz.fnusa.15_1_95th percentile of acceleration (in-air)    float64
w.cz.fnusa.16_1_95th percentile of acceleration (in-air)    float64
w.cz.fnusa.17_1_95th percentile of acceleration (in-air)    float64
                                                             ...   
memory z-score                                              float64
visuo-spatial z-score                                       float64
attention z-score                                           float64
executive function z-score                                  float64
GDS                                                         float64
Length: 463, dtype: object

In [462]:

from scipy import stats
import numpy as np

meta_cols=["age",
         #   "gender",
            "MOCA",
            "education type",
            "education length",
            "memory z-score",
            "visuo-spatial z-score",
            "attention z-score",
            "executive function z-score",
            "GDS"]

feature_cols = clinical_hf_emb_df_complet_lbl.drop(columns=['ID', 'diagnosis','gender'] + meta_cols).columns.tolist()


def correlation_to_df(df, feature_cols, meta_cols, corr_type='spearman'):
    from scipy import stats
    corr_results = []
    for feature in feature_cols:
        for meta in meta_cols:
            feature_data = pd.to_numeric(df[feature], errors='coerce')
            meta_data = pd.to_numeric(df[meta], errors='coerce')
            if corr_type == 'spearman':
                corr, p_value = stats.spearmanr(feature_data, meta_data, nan_policy='omit')
            elif corr_type == 'pearson':
                corr, p_value = stats.pearsonr(feature_data.dropna(), meta_data.dropna())
            else:
                raise ValueError("corr_type must be 'spearman' or 'pearson'")
            corr_results.append({
                'feature': feature,
                'meta_variable': meta,
                'correlation': corr,
                'p_value': p_value
            })
    corr_df = pd.DataFrame(corr_results)
    return corr_df


In [463]:
import numpy as np
import pandas as pd
from scipy import stats

def safe_spearman(x, y, min_n=3):
    """
    x, y: 1D array-like
    returns (rho, p, n_used)
    """
    x = pd.to_numeric(pd.Series(x), errors="coerce")
    y = pd.to_numeric(pd.Series(y), errors="coerce")

    mask = x.notna() & y.notna()
    x2 = x[mask].values
    y2 = y[mask].values
    n = len(x2)

    if n < min_n:
        return np.nan, np.nan, n

    # constant vectors -> undefined correlation
    if np.all(x2 == x2[0]) or np.all(y2 == y2[0]):
        return np.nan, np.nan, n

    rho, p = stats.spearmanr(x2, y2)
    # spearmanr can still return nan if ties/degenerate
    if not np.isfinite(rho):
        rho, p = np.nan, np.nan
    return rho, p, n


def correlation_to_df(df, feature_cols, meta_cols, corr_type="spearman", min_n=3):
    rows = []
    for feat in feature_cols:
        feature_data = df[feat]

        for meta in meta_cols:
            meta_data = df[meta]

            if corr_type == "spearman":
                corr, p_value, n_used = safe_spearman(feature_data, meta_data, min_n=min_n)
            elif corr_type == "pearson":
                # paired dropna for pearson too
                x = pd.to_numeric(feature_data, errors="coerce")
                y = pd.to_numeric(meta_data, errors="coerce")
                mask = x.notna() & y.notna()
                x2, y2 = x[mask].values, y[mask].values
                n_used = len(x2)
                if n_used < min_n or np.all(x2 == x2[0]) or np.all(y2 == y2[0]):
                    corr, p_value = np.nan, np.nan
                else:
                    corr, p_value = stats.pearsonr(x2, y2)
            else:
                raise ValueError("corr_type must be 'spearman' or 'pearson'")

            rows.append({
                "feature": feat,
                "meta": meta,
                "corr": corr,
                "p_value": p_value,
                "n_used": n_used
            })

    return pd.DataFrame(rows)


In [575]:
clinical_hf_emb_df_complet_lbl

,Unnamed: 0,ID,w.cz.fnusa.15_1_95th percentile of acceleration (in-air),w.cz.fnusa.16_1_95th percentile of acceleration (in-air),w.cz.fnusa.17_1_95th percentile of acceleration (in-air),w.cz.fnusa.18_1_95th percentile of acceleration (in-air),w.cz.fnusa.19_1_95th percentile of acceleration (in-air),w.cz.fnusa.9_1_95th percentile of acceleration (in-air),w.cz.fnusa.15_1_95th percentile of acceleration (on-surface),w.cz.fnusa.16_1_95th percentile of acceleration (on-surface),...,age,gender,MOCA,education type,education length,memory z-score,visuo-spatial z-score,attention z-score,executive function z-score,GDS
0,0,NaN,1382.947668,2062.000865,2892.979665,2192.046228,2013.070361,2235.580284,586.734694,1627.424458,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,NaN,2184.038058,2960.491263,2833.151798,NaN,2643.941418,2704.362771,1705.984308,1274.972304,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,NaN,2129.216623,3373.868306,1686.794432,NaN,3212.659823,3435.479391,1657.058812,1573.868658,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,NaN,3973.130385,3650.787752,2810.386562,NaN,3278.226429,2390.344711,2226.888919,2314.558206,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,NaN,973.083976,1194.677753,2163.527710,NaN,1601.144846,1533.402248,581.075743,1754.342814,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,261,NaN,1771.638967,2223.266011,2726.863796,NaN,NaN,NaN,1248.437707,2379.581423,...,55.0,1.0,27.0,2.0,12.0,-0.43,-0.25,0.00,0.22,13.0
262,262,NaN,1587.495217,3116.456725,5244.552340,NaN,3146.502315,3071.576595,1670.497532,5487.155540,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
263,263,NaN,1529.780961,2846.530626,3170.341400,NaN,2702.591544,1901.945515,914.354564,4092.897902,...,70.0,0.0,26.0,3.0,13.0,-0.66,1.08,-0.33,-1.17,4.0
264,264,NaN,1875.912627,2772.534223,3176.331668,NaN,2088.759949,2417.683099,775.466348,1565.536158,...,75.0,1.0,14.0,NaN,NaN,NaN,NaN,NaN,NaN,19.0


In [465]:
df_hf_clinical_corr = correlation_to_df(clinical_hf_emb_df_complet_lbl, feature_cols, meta_cols, corr_type='spearman')

In [466]:
df_hf_clinical_corr

,feature,meta,corr,p_value,n_used
0,Unnamed: 0,age,0.221721,0.027412,99
1,Unnamed: 0,MOCA,-0.371650,0.000178,97
2,Unnamed: 0,education type,-0.337577,0.000769,96
3,Unnamed: 0,education length,-0.358016,0.000342,96
4,Unnamed: 0,memory z-score,0.102028,0.325190,95
...,...,...,...,...,...
4054,w.cz.fnusa.1_1_zero-crossing rate of spiral,memory z-score,0.142506,0.168318,95
4055,w.cz.fnusa.1_1_zero-crossing rate of spiral,visuo-spatial z-score,0.122164,0.240807,94
4056,w.cz.fnusa.1_1_zero-crossing rate of spiral,attention z-score,0.209750,0.041343,95
4057,w.cz.fnusa.1_1_zero-crossing rate of spiral,executive function z-score,0.268764,0.008450,95


In [467]:
df_hf_clinical_corr = df_hf_clinical_corr.drop(df_hf_clinical_corr.columns[-1], axis=1) 

In [468]:
df_hf_clinical_corr

,feature,meta,corr,p_value
0,Unnamed: 0,age,0.221721,0.027412
1,Unnamed: 0,MOCA,-0.371650,0.000178
2,Unnamed: 0,education type,-0.337577,0.000769
3,Unnamed: 0,education length,-0.358016,0.000342
4,Unnamed: 0,memory z-score,0.102028,0.325190
...,...,...,...,...
4054,w.cz.fnusa.1_1_zero-crossing rate of spiral,memory z-score,0.142506,0.168318
4055,w.cz.fnusa.1_1_zero-crossing rate of spiral,visuo-spatial z-score,0.122164,0.240807
4056,w.cz.fnusa.1_1_zero-crossing rate of spiral,attention z-score,0.209750,0.041343
4057,w.cz.fnusa.1_1_zero-crossing rate of spiral,executive function z-score,0.268764,0.008450


In [469]:
# Create separate pivots for correlation and p-value
#df_corr_pivot = df_hf_clinical_corr.pivot(
#    index='feature',
#    columns='meta',
#    values='corr'
#)

df_pval_pivot = df_hf_clinical_corr.pivot(
    index='feature',
    columns='meta',
    values='p_value'
)

In [470]:
df_pval_pivot[df_pval_pivot < 0.05].count()

meta
GDS                           70
MOCA                          66
age                           43
attention z-score             61
education length              53
education type                53
executive function z-score    24
memory z-score                11
visuo-spatial z-score         69
dtype: int64

# FDR correction

In [471]:

import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

def fdr_correction_matrix(df_pvals: pd.DataFrame, alpha: float = 0.05, method: str = "fdr_bh"):
    df = df_pvals.copy()

    # Only numeric columns (skip "feature")
    numeric_cols = df.select_dtypes(include=[float, int]).columns
    arr = df[numeric_cols].to_numpy(dtype=float)

    # Mask NaNs
    mask_na = np.isnan(arr)

    # Take only valid p-values (flatten)
    pvals_flat = arr[~mask_na]

    # FDR
    reject, pvals_corr, _, _ = multipletests(pvals_flat, alpha=alpha, method=method)

    # Prepare output arrays
    arr_fdr = np.full_like(arr, np.nan, dtype=float)
    arr_sig = np.zeros_like(arr, dtype=bool)

    # Fill corrected values
    arr_fdr[~mask_na] = pvals_corr
    arr_sig[~mask_na] = reject

    # Back to DataFrames
    df_fdr = df.copy()
    df_sig = df.copy()

    df_fdr[numeric_cols] = arr_fdr
    df_sig[numeric_cols] = arr_sig

    return df_fdr, df_sig


In [472]:

df_fdr, df_sig = fdr_correction_matrix(df_pval_pivot, alpha=0.05)

#Show significant (feature, metadata) pairs
significant_pairs = (
    df_sig
         .iloc[:, 1:]         # skip the 'feature' column
         .stack()             # long format
         .loc[lambda s: s]    # keep only True
         .index
         .to_frame(name=["feature", "metadata"])
)
##
significant_pairs


feature  \
feature                                            meta                                                                       
Unnamed: 0                                         MOCA                                                          Unnamed: 0   
                                                   visuo-spatial z-score                                         Unnamed: 0   
w.cz.fnusa.17_1_slope of velocity (in-air)         visuo-spatial z-score         w.cz.fnusa.17_1_slope of velocity (in-air)   
w.cz.fnusa.17_1_slope of vertical velocity (in-... visuo-spatial z-score  w.cz.fnusa.17_1_slope of vertical velocity (in...   

                                                                                       metadata  
feature                                            meta                                          
Unnamed: 0                                         MOCA                                    MOCA  
                                                   visuo-spatial z-score  visuo-spatial z-score  
w.cz.fnusa.17_1_slope of velocity (in-air)         visuo-spatial z-score  visuo-spatial z-score  
w.cz.fnusa.17_1_slope of vertical velocity (in-... visuo-spatial z-score  visuo-spatial z-score

In [473]:

df_after_fdr = df_fdr[df_sig]

In [474]:

df_after_fdr.dropna(how='all', inplace=True)
df_after_fdr

meta,GDS,MOCA,age,attention z-score,education length,education type,executive function z-score,memory z-score,visuo-spatial z-score
feature,,,,,,,,,
Unnamed: 0,0.000039,0.044494,NaN,NaN,NaN,NaN,NaN,NaN,0.032332
w.cz.fnusa.16_1_95th percentile of acceleration (on-surface),0.036790,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
w.cz.fnusa.16_1_95th percentile of horizontal acceleration (on-surface),0.044494,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
w.cz.fnusa.16_1_95th percentile of velocity (on-surface),0.032332,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
w.cz.fnusa.16_1_95th percentile of vertical acceleration (on-surface),0.040616,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
w.cz.fnusa.16_1_95th percentile of vertical velocity (on-surface),0.036790,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
w.cz.fnusa.17_1_95th percentile of acceleration (on-surface),0.032332,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
w.cz.fnusa.17_1_95th percentile of angular velocity (on-surface),0.044494,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
w.cz.fnusa.17_1_95th percentile of horizontal acceleration (on-surface),0.036790,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Task-wise Dataframe dictionaries creation

- temp_emb_df_complet_lbl_2
- freq_emb_df_complet_lbl_2
- comb_emb_df_complet_lbl_2
- hf_emb_df_complet_lbl_2 

In [730]:
def task_wise_df_creation(df, task_name):
    task_df = df[df['file_path'].str.contains(task_name)].reset_index(drop=True)
    return task_df

In [788]:

task_list = ['1_1', '9_1', '15_1', '16_1', '17_1', '18_1', '19_1']

task_freq_dfs = {task: task_wise_df_creation(freq_emb_df_complet_lbl, task) for task in task_list}
task_temp_dfs = {task: task_wise_df_creation(temp_emb_df_complet_lbl, task) for task in task_list}
task_comb_dfs = {task: task_wise_df_creation(comb_emb_df_complet_lbl, task) for task in task_list}

In [789]:

task_list = ['1_1', '9_1', '15_1', '16_1', '17_1', '18_1', '19_1']

task_hf_dfs = {
    task: hf_emb_df_complet_lbl.loc[
        :,
        hf_emb_df_complet_lbl.columns.astype(str).str.contains(task, case=False) |
        hf_emb_df_complet_lbl.columns.isin(['diagnosis', 'ID'])
    ].copy()
    for task in task_list
}

## MCI-LBD vs HC

In [790]:

task_freq_dfs_lbd_hc = {k: v[~v["diagnosis"].isin([1, 2, 4])].reset_index(drop=True) for k, v in task_freq_dfs.items()}
task_temp_dfs_lbd_hc = {k: v[~v["diagnosis"].isin([1, 2, 4])].reset_index(drop=True) for k, v in task_temp_dfs.items()}
task_comb_dfs_lbd_hc = {k: v[~v["diagnosis"].isin([1, 2, 4])].reset_index(drop=True) for k, v in task_comb_dfs.items()}
task_hf_dfs_lbd_hc = {k: v[~v["diagnosis"].isin([1, 2, 4])].reset_index(drop=True) for k, v in task_hf_dfs.items()}

## MCI-AD vs HC

In [791]:
task_freq_dfs_ad_hc = {k: v[~v["diagnosis"].isin([1, 3, 4])].reset_index(drop=True) for k, v in task_freq_dfs.items()}
task_temp_dfs_ad_hc = {k: v[~v["diagnosis"].isin([1, 3, 4])].reset_index(drop=True) for k, v in task_temp_dfs.items()}
task_comb_dfs_ad_hc = {k: v[~v["diagnosis"].isin([1, 3, 4])].reset_index(drop=True) for k, v in task_comb_dfs.items()}
task_hf_dfs_ad_hc = {k: v[~v["diagnosis"].isin([1, 3, 4])].reset_index(drop=True) for k, v in task_hf_dfs.items()}

## MCI-PD vs HC

In [792]:

task_freq_dfs_pd_hc = {k: v[~v["diagnosis"].isin([1, 3, 2])].reset_index(drop=True) for k, v in task_freq_dfs.items()}
task_temp_dfs_pd_hc = {k: v[~v["diagnosis"].isin([1, 3, 2])].reset_index(drop=True) for k, v in task_temp_dfs.items()}
task_comb_dfs_pd_hc = {k: v[~v["diagnosis"].isin([1, 3, 2])].reset_index(drop=True) for k, v in task_comb_dfs.items()}
task_hf_dfs_pd_hc = {k: v[~v["diagnosis"].isin([1, 3, 2])].reset_index(drop=True) for k, v in task_hf_dfs.items()}

## MCI-LBD vs MCI-AD

In [793]:

task_freq_dfs_ad_lbd = {k: v[~v["diagnosis"].isin([1, 0, 4])].reset_index(drop=True) for k, v in task_freq_dfs.items()}
task_temp_dfs_ad_lbd = {k: v[~v["diagnosis"].isin([1, 0, 4])].reset_index(drop=True) for k, v in task_temp_dfs.items()}
task_comb_dfs_ad_lbd = {k: v[~v["diagnosis"].isin([1, 0, 4])].reset_index(drop=True) for k, v in task_comb_dfs.items()}
task_hf_dfs_ad_lbd = {k: v[~v["diagnosis"].isin([1, 0, 4])].reset_index(drop=True) for k, v in task_hf_dfs.items()}

In [794]:
task_freq_dfs_lbd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_freq_dfs_lbd_hc.items()}
task_temp_dfs_lbd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_temp_dfs_lbd_hc.items()}
task_comb_dfs_lbd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_comb_dfs_lbd_hc.items()}

In [795]:

task_freq_dfs_ad_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_freq_dfs_ad_hc.items()}
task_temp_dfs_ad_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_temp_dfs_ad_hc.items()}
task_comb_dfs_ad_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_comb_dfs_ad_hc.items()}

In [796]:
task_freq_dfs_pd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_freq_dfs_pd_hc.items()}
task_temp_dfs_pd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_temp_dfs_pd_hc.items()}
task_comb_dfs_pd_hc = {k: v.drop(columns=['label', 'file_path']) for k, v in task_comb_dfs_pd_hc.items()}

In [797]:

task_freq_dfs_ad_lbd = {k: v.drop(columns=['label', 'file_path']) for k, v in task_freq_dfs_ad_lbd.items()}
task_temp_dfs_ad_lbd = {k: v.drop(columns=['label', 'file_path']) for k, v in task_temp_dfs_ad_lbd.items()}
task_comb_dfs_ad_lbd = {k: v.drop(columns=['label', 'file_path']) for k, v in task_comb_dfs_ad_lbd.items()}

In [798]:
for dfs in (task_freq_dfs_lbd_hc, task_temp_dfs_lbd_hc, task_comb_dfs_lbd_hc, task_hf_dfs_lbd_hc):
    
    for k, v in dfs.items():
        if "diagnosis" in v.columns:
            v["diagnosis"] = v["diagnosis"].replace({3.0: 1, 0.0: 0})


C:\Users\tenNovy\AppData\Local\Temp\ipykernel_1420\2570176340.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  v["diagnosis"] = v["diagnosis"].replace({3.0: 1, 0.0: 0})


In [799]:
for dfs in (task_freq_dfs_ad_hc, task_temp_dfs_ad_hc, task_comb_dfs_ad_hc, task_hf_dfs_ad_hc):
    
    for k, v in dfs.items():
        if "diagnosis" in v.columns:
            v["diagnosis"] = v["diagnosis"].replace({2.0: 1, 0.0: 0})


C:\Users\tenNovy\AppData\Local\Temp\ipykernel_1420\856415710.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  v["diagnosis"] = v["diagnosis"].replace({2.0: 1, 0.0: 0})


In [800]:

for dfs in (task_freq_dfs_ad_lbd, task_temp_dfs_ad_lbd, task_comb_dfs_ad_lbd, task_hf_dfs_ad_lbd):
    
    for k, v in dfs.items():
        if "diagnosis" in v.columns:
            v["diagnosis"] = v["diagnosis"].replace({3.0: 1, 2.0: 0})

C:\Users\tenNovy\AppData\Local\Temp\ipykernel_1420\676543069.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  v["diagnosis"] = v["diagnosis"].replace({3.0: 1, 2.0: 0})


In [801]:
for dfs in (task_freq_dfs_pd_hc, task_temp_dfs_pd_hc, task_comb_dfs_pd_hc, task_hf_dfs_pd_hc):
    
    for k, v in dfs.items():
        if "diagnosis" in v.columns:
            v["diagnosis"] = v["diagnosis"].replace({4.0: 1, 0.0: 0})


C:\Users\tenNovy\AppData\Local\Temp\ipykernel_1420\1503130897.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  v["diagnosis"] = v["diagnosis"].replace({4.0: 1, 0.0: 0})


In [614]:
task_freq_dfs_ad_hc.get("1_1")

,subject,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383,diagnosis
0,COBEN-WTABLET-AS-HCD05,-0.053105,0.043560,0.012405,-0.010514,-0.025581,-0.016981,-0.065453,0.058739,-0.006989,...,-0.088822,-0.022320,-0.026403,-0.013258,0.028018,-0.070962,0.068714,0.030881,0.056124,0.0
1,COBEN-WTABLET-DM-AD15,-0.051845,0.043946,0.011904,-0.010953,-0.025803,-0.017803,-0.065916,0.059028,-0.006370,...,-0.088997,-0.022654,-0.026079,-0.012855,0.027741,-0.071593,0.069436,0.030476,0.055148,1.0
2,COBEN-WTABLET-DS-HC36,-0.053084,0.043164,0.011447,-0.010110,-0.025780,-0.016541,-0.065701,0.058689,-0.006781,...,-0.089358,-0.021474,-0.026835,-0.013402,0.029021,-0.070640,0.069674,0.030071,0.056516,0.0
3,COBEN-WTABLET-DV-AD04,-0.054663,0.041894,0.012110,-0.010013,-0.024074,-0.016389,-0.065150,0.055417,-0.007816,...,-0.088301,-0.022028,-0.027626,-0.014927,0.029764,-0.070136,0.065817,0.032516,0.055742,1.0
4,COBEN-WTABLET-EH-HC35,-0.053226,0.043825,0.011746,-0.010794,-0.025438,-0.016439,-0.065685,0.058769,-0.006327,...,-0.089707,-0.021616,-0.027196,-0.013121,0.028772,-0.070659,0.068586,0.030430,0.057629,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,HC-32#1,-0.053320,0.042233,0.012059,-0.010998,-0.025761,-0.017718,-0.065758,0.058969,-0.007252,...,-0.089658,-0.022199,-0.026838,-0.013825,0.027848,-0.070209,0.067426,0.028689,0.056591,0.0
71,HC-33#1,-0.053044,0.043273,0.011899,-0.010504,-0.025557,-0.016242,-0.065716,0.058804,-0.006392,...,-0.089575,-0.021549,-0.027259,-0.013552,0.028940,-0.070731,0.067962,0.030493,0.057406,0.0
72,HC-34#1,-0.052794,0.043555,0.012379,-0.010400,-0.025120,-0.016714,-0.065440,0.058418,-0.007094,...,-0.089103,-0.021499,-0.026716,-0.013113,0.028311,-0.071051,0.069072,0.030863,0.056335,0.0
73,HC-4#1,-0.052568,0.043872,0.011817,-0.011168,-0.025485,-0.017019,-0.065562,0.058993,-0.006118,...,-0.088956,-0.021848,-0.026827,-0.012976,0.028284,-0.071380,0.068624,0.030577,0.057050,1.0


In [802]:

def miss_val_clean(df, percent_threshold=0.8):
    """
    Cleans the DataFrame by removing columns with more than the specified percentage of missing values.
    
    Parameters
    ----------
    df : pd.DataFrame
        The input DataFrame to be cleaned.
    percent_threshold : float
        The maximum allowed percentage of missing values in a column (between 0 and 1).
        
    Returns
    -------
    pd.DataFrame
        The cleaned DataFrame with columns exceeding the missing value threshold removed.
    """
    df_cleaned = df.copy()
    df_cleaned = df_cleaned.loc[:, df_cleaned.isnull().mean() <= percent_threshold]
    
    cols_to_convert = df_cleaned.columns[1:-1]

    df_cleaned[cols_to_convert] = (
        df_cleaned[cols_to_convert]
        .apply(lambda col: pd.to_numeric(col.astype(str).str.replace(',', '.'), errors='coerce'))
    )

    df_cleaned = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))
    return df_cleaned

In [803]:

task_hf_dfs_clean_lbd_hc = {k: miss_val_clean(v, percent_threshold=0.8) for k, v in task_hf_dfs_lbd_hc.items()}
task_hf_dfs_clean_ad_hc = {k: miss_val_clean(v, percent_threshold=0.8) for k, v in task_hf_dfs_ad_hc.items()}
task_hf_dfs_clean_ad_lbd = {k: miss_val_clean(v, percent_threshold=0.8) for k, v in task_hf_dfs_ad_lbd.items()}
task_hf_dfs_clean_pd_hc = {k: miss_val_clean(v, percent_threshold=0.8) for k, v in task_hf_dfs_pd_hc.items()}

In [804]:

# Use this one 
def merge_on_subject_agg_right(df1, df2):
    df1 = df1.copy(); df2 = df2.copy()
    if 'subject' not in df1 and 'ID' in df1: df1 = df1.rename(columns={'ID':'subject'})
    if 'subject' not in df2 and 'ID' in df2: df2 = df2.rename(columns={'ID':'subject'})
    num_cols = df2.select_dtypes('number').columns.tolist()
    agg = {c:'first' for c in df2.columns if c not in num_cols and c!='subject'}
    agg.update({c:'mean' for c in num_cols})
    df2 = df2.groupby('subject', as_index=False).agg(agg)
    return pd.merge(df1, df2, on='subject', how='inner', validate='many_to_one', sort=False)


### MCI-LBD vs HC

In [805]:
task_hf_compensated_freq_emb_lbl_lbd_hc = {
    k: merge_on_subject_agg_right(task_freq_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_freq_dfs_lbd_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

In [806]:

task_hf_compensated_temp_emb_lbl_lbd_hc = {
    k: merge_on_subject_agg_right(task_temp_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_temp_dfs_lbd_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

In [807]:

task_hf_compensated_comb_emb_lbl_lbd_hc = {
    k: merge_on_subject_agg_right(task_comb_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_comb_dfs_lbd_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

In [808]:
task_hf_comb_emb_lbl_lbd_hc = {
    k: merge_on_subject_agg_right(task_comb_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_comb_dfs_lbd_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

In [809]:
task_hf_freq_emb_lbl_lbd_hc = {
    k: merge_on_subject_agg_right(task_freq_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_freq_dfs_lbd_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

In [810]:
task_hf_temp_emb_lbl_lbd_hc = {
    k: merge_on_subject_agg_right(task_temp_dfs_lbd_hc[k], task_hf_dfs_clean_lbd_hc[k])
    for k in task_temp_dfs_lbd_hc.keys() & task_hf_dfs_clean_lbd_hc.keys()
}

### MCI-AD vs HC

In [811]:
### MCI-AD vs HC
task_hf_compensated_freq_emb_lbl_ad_hc = {
    k: merge_on_subject_agg_right(task_freq_dfs_ad_hc[k], task_hf_dfs_clean_ad_hc[k])
    for k in task_freq_dfs_ad_hc.keys() & task_hf_dfs_clean_ad_hc.keys()
}

task_hf_compensated_temp_emb_lbl_ad_hc = {
    k: merge_on_subject_agg_right(task_temp_dfs_ad_hc[k], task_hf_dfs_clean_ad_hc[k])
    for k in task_temp_dfs_ad_hc.keys() & task_hf_dfs_clean_ad_hc.keys()
}

task_hf_compensated_comb_emb_lbl_ad_hc = {
    k: merge_on_subject_agg_right(task_comb_dfs_ad_hc[k], task_hf_dfs_clean_ad_hc[k])
    for k in task_comb_dfs_ad_hc.keys() & task_hf_dfs_clean_ad_hc.keys()
}
task_hf_comb_emb_lbl_ad_hc= {
    k: merge_on_subject_agg_right(task_comb_dfs_ad_hc[k], task_hf_dfs_clean_ad_hc[k])
    for k in task_comb_dfs_ad_hc.keys() & task_hf_dfs_clean_ad_hc.keys()
}
task_hf_freq_emb_lbl_ad_hc = {
    k: merge_on_subject_agg_right(task_freq_dfs_ad_hc[k], task_hf_dfs_clean_ad_hc[k])
    for k in task_freq_dfs_ad_hc.keys() & task_hf_dfs_clean_ad_hc.keys()
}
task_hf_temp_emb_lbl_ad_hc = {
    k: merge_on_subject_agg_right(task_temp_dfs_ad_hc[k], task_hf_dfs_clean_ad_hc[k])
    for k in task_temp_dfs_ad_hc.keys() & task_hf_dfs_clean_ad_hc.keys()
}

In [623]:
task_hf_compensated_freq_emb_lbl_ad_hc.get("1_1")

,subject,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,w.cz.fnusa.1_1_slope of duration of pen stops,w.cz.fnusa.1_1_slope of horizontal velocity (on-surface),w.cz.fnusa.1_1_slope of pressure,w.cz.fnusa.1_1_slope of velocity (on-surface),w.cz.fnusa.1_1_slope of vertical velocity (on-surface),w.cz.fnusa.1_1_spiral precision index,w.cz.fnusa.1_1_tightness of spiral,w.cz.fnusa.1_1_variability of spiral width,w.cz.fnusa.1_1_zero-crossing rate of spiral,diagnosis_y
0,COBEN-WTABLET-AS-HCD05,-0.053105,0.043560,0.012405,-0.010514,-0.025581,-0.016981,-0.065453,0.058739,-0.006989,...,-0.056107,0.041549,0.000237,0.066626,0.042579,26.241991,1.965061,0.081824,4.293521,0.0
1,COBEN-WTABLET-DM-AD15,-0.051845,0.043946,0.011904,-0.010953,-0.025803,-0.017803,-0.065916,0.059028,-0.006370,...,0.041338,0.012307,0.000107,0.017672,0.010502,18.809973,1.891182,0.090616,4.502542,1.0
2,COBEN-WTABLET-DS-HC36,-0.053084,0.043164,0.011447,-0.010110,-0.025780,-0.016541,-0.065701,0.058689,-0.006781,...,-1.300261,0.010903,0.000142,0.018111,0.011886,21.863560,1.624943,0.144086,5.076851,0.0
3,COBEN-WTABLET-DV-AD04,-0.054663,0.041894,0.012110,-0.010013,-0.024074,-0.016389,-0.065150,0.055417,-0.007816,...,0.015836,0.006629,0.000076,0.008595,0.004086,16.379484,2.371073,0.099080,7.525870,1.0
4,COBEN-WTABLET-EH-HC35,-0.053226,0.043825,0.011746,-0.010794,-0.025438,-0.016439,-0.065685,0.058769,-0.006327,...,-0.056107,0.011419,-0.000003,0.018186,0.011020,11.250131,2.830892,0.111882,4.425100,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,HC-32#1,-0.053320,0.042233,0.012059,-0.010998,-0.025761,-0.017718,-0.065758,0.058969,-0.007252,...,-0.056107,0.154359,0.000451,0.209169,0.098028,24.558513,1.373170,0.134250,6.290672,0.0
71,HC-33#1,-0.053044,0.043273,0.011899,-0.010504,-0.025557,-0.016242,-0.065716,0.058804,-0.006392,...,-0.056107,0.018290,0.000127,0.026143,0.014093,15.589412,1.505466,0.077682,5.936073,0.0
72,HC-34#1,-0.052794,0.043555,0.012379,-0.010400,-0.025120,-0.016714,-0.065440,0.058418,-0.007094,...,-0.056107,0.013007,0.000044,0.018650,0.010024,19.916698,1.272528,0.150220,5.373593,0.0
73,HC-4#1,-0.052568,0.043872,0.011817,-0.011168,-0.025485,-0.017019,-0.065562,0.058993,-0.006118,...,-0.056107,0.003749,-0.000005,0.005173,0.002177,13.384290,1.909837,0.082338,4.804701,1.0


### MCI-LBD vs MCI-AD

In [812]:
### MCI-LBD vs MCI-LBD
task_hf_compensated_freq_emb_lbl_ad_lbd = {
    k: merge_on_subject_agg_right(task_freq_dfs_ad_lbd[k], task_hf_dfs_clean_ad_lbd[k])
    for k in task_freq_dfs_ad_lbd.keys() & task_hf_dfs_clean_ad_lbd.keys()
}

task_hf_compensated_temp_emb_lbl_ad_lbd = {
    k: merge_on_subject_agg_right(task_temp_dfs_ad_lbd[k], task_hf_dfs_clean_ad_lbd[k])
    for k in task_temp_dfs_ad_lbd.keys() & task_hf_dfs_clean_ad_lbd.keys()
}

task_hf_compensated_comb_emb_lbl_ad_lbd = {
    k: merge_on_subject_agg_right(task_comb_dfs_ad_lbd[k], task_hf_dfs_clean_ad_lbd[k])
    for k in task_comb_dfs_ad_lbd.keys() & task_hf_dfs_clean_ad_lbd.keys()
}
task_hf_comb_emb_lbl_ad_lbd= {
    k: merge_on_subject_agg_right(task_comb_dfs_ad_lbd[k], task_hf_dfs_clean_ad_lbd[k])
    for k in task_comb_dfs_ad_lbd.keys() & task_hf_dfs_clean_ad_lbd.keys()
}
task_hf_freq_emb_lbl_ad_lbd = {
    k: merge_on_subject_agg_right(task_freq_dfs_ad_lbd[k], task_hf_dfs_clean_ad_lbd[k])
    for k in task_freq_dfs_ad_lbd.keys() & task_hf_dfs_clean_ad_lbd.keys()
}
task_hf_temp_emb_lbl_ad_lbd = {
    k: merge_on_subject_agg_right(task_temp_dfs_ad_lbd[k], task_hf_dfs_clean_ad_lbd[k])
    for k in task_temp_dfs_ad_lbd.keys() & task_hf_dfs_clean_ad_lbd.keys()
}

### PD vs HC

In [813]:

### MCI-AD vs HC
task_hf_compensated_freq_emb_lbl_pd_hc = {
    k: merge_on_subject_agg_right(task_freq_dfs_pd_hc[k], task_hf_dfs_clean_pd_hc[k])
    for k in task_freq_dfs_pd_hc.keys() & task_hf_dfs_clean_pd_hc.keys()
}

task_hf_compensated_temp_emb_lbl_pd_hc = {
    k: merge_on_subject_agg_right(task_temp_dfs_pd_hc[k], task_hf_dfs_clean_pd_hc[k])
    for k in task_temp_dfs_pd_hc.keys() & task_hf_dfs_clean_pd_hc.keys()
}

task_hf_compensated_comb_emb_lbl_pd_hc = {
    k: merge_on_subject_agg_right(task_comb_dfs_pd_hc[k], task_hf_dfs_clean_pd_hc[k])
    for k in task_comb_dfs_pd_hc.keys() & task_hf_dfs_clean_pd_hc.keys()
}
task_hf_comp_emb_lbl_pd_hc= {
    k: merge_on_subject_agg_right(task_comb_dfs_pd_hc[k], task_hf_dfs_clean_pd_hc[k])
    for k in task_comb_dfs_pd_hc.keys() & task_hf_dfs_clean_pd_hc.keys()
}
task_hf_freq_emb_lbl_pd_hc = {
    k: merge_on_subject_agg_right(task_freq_dfs_pd_hc[k], task_hf_dfs_clean_pd_hc[k])
    for k in task_freq_dfs_pd_hc.keys() & task_hf_dfs_clean_pd_hc.keys()
}
task_hf_temp_emb_lbl_pd_hc = {
    k: merge_on_subject_agg_right(task_temp_dfs_pd_hc[k], task_hf_dfs_clean_pd_hc[k])
    for k in task_temp_dfs_pd_hc.keys() & task_hf_dfs_clean_pd_hc.keys()
}

In [814]:
def x_y_split(df, split_param):
    if split_param is None or split_param == "diagnosis_y":
        #embedding_cols = df[1:-1]
        X = df.iloc[:,1:-2].values
        y = df["diagnosis_y"].values
    elif split_param != None: 
        embedding_cols = [c for c in df.columns if c.startswith(split_param)]
        #X = df[embedding_cols].values
        X = df.iloc[:,1:-2].values
        y = df["diagnosis"].values
    
    return X, y



In [815]:
def remove_leaking_labels(df, label_string = 'diagnosis_x'):
    df_cleaned = df.copy()
    df_cleaned = df_cleaned.drop(label_string, axis = 1)
    return df_cleaned

In [816]:
task_hf_compensated_freq_emb_lbl_ad_hc.get("1_1")

,subject,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,w.cz.fnusa.1_1_slope of duration of pen stops,w.cz.fnusa.1_1_slope of horizontal velocity (on-surface),w.cz.fnusa.1_1_slope of pressure,w.cz.fnusa.1_1_slope of velocity (on-surface),w.cz.fnusa.1_1_slope of vertical velocity (on-surface),w.cz.fnusa.1_1_spiral precision index,w.cz.fnusa.1_1_tightness of spiral,w.cz.fnusa.1_1_variability of spiral width,w.cz.fnusa.1_1_zero-crossing rate of spiral,diagnosis_y
0,COBEN-WTABLET-AS-HCD05,-0.053105,0.043560,0.012405,-0.010514,-0.025581,-0.016981,-0.065453,0.058739,-0.006989,...,-0.056107,0.041549,0.000237,0.066626,0.042579,26.241991,1.965061,0.081824,4.293521,0.0
1,COBEN-WTABLET-DM-AD15,-0.051845,0.043946,0.011904,-0.010953,-0.025803,-0.017803,-0.065916,0.059028,-0.006370,...,0.041338,0.012307,0.000107,0.017672,0.010502,18.809973,1.891182,0.090616,4.502542,1.0
2,COBEN-WTABLET-DS-HC36,-0.053084,0.043164,0.011447,-0.010110,-0.025780,-0.016541,-0.065701,0.058689,-0.006781,...,-1.300261,0.010903,0.000142,0.018111,0.011886,21.863560,1.624943,0.144086,5.076851,0.0
3,COBEN-WTABLET-DV-AD04,-0.054663,0.041894,0.012110,-0.010013,-0.024074,-0.016389,-0.065150,0.055417,-0.007816,...,0.015836,0.006629,0.000076,0.008595,0.004086,16.379484,2.371073,0.099080,7.525870,1.0
4,COBEN-WTABLET-EH-HC35,-0.053226,0.043825,0.011746,-0.010794,-0.025438,-0.016439,-0.065685,0.058769,-0.006327,...,-0.056107,0.011419,-0.000003,0.018186,0.011020,11.250131,2.830892,0.111882,4.425100,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,HC-32#1,-0.053320,0.042233,0.012059,-0.010998,-0.025761,-0.017718,-0.065758,0.058969,-0.007252,...,-0.056107,0.154359,0.000451,0.209169,0.098028,24.558513,1.373170,0.134250,6.290672,0.0
71,HC-33#1,-0.053044,0.043273,0.011899,-0.010504,-0.025557,-0.016242,-0.065716,0.058804,-0.006392,...,-0.056107,0.018290,0.000127,0.026143,0.014093,15.589412,1.505466,0.077682,5.936073,0.0
72,HC-34#1,-0.052794,0.043555,0.012379,-0.010400,-0.025120,-0.016714,-0.065440,0.058418,-0.007094,...,-0.056107,0.013007,0.000044,0.018650,0.010024,19.916698,1.272528,0.150220,5.373593,0.0
73,HC-4#1,-0.052568,0.043872,0.011817,-0.011168,-0.025485,-0.017019,-0.065562,0.058993,-0.006118,...,-0.056107,0.003749,-0.000005,0.005173,0.002177,13.384290,1.909837,0.082338,4.804701,1.0


In [641]:
task_hf_compensated_freq_emb_lbl_ad_hc.get("1_1")

,subject,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,w.cz.fnusa.1_1_slope of duration of pen stops,w.cz.fnusa.1_1_slope of horizontal velocity (on-surface),w.cz.fnusa.1_1_slope of pressure,w.cz.fnusa.1_1_slope of velocity (on-surface),w.cz.fnusa.1_1_slope of vertical velocity (on-surface),w.cz.fnusa.1_1_spiral precision index,w.cz.fnusa.1_1_tightness of spiral,w.cz.fnusa.1_1_variability of spiral width,w.cz.fnusa.1_1_zero-crossing rate of spiral,diagnosis_y
0,COBEN-WTABLET-AS-HCD05,-0.053105,0.043560,0.012405,-0.010514,-0.025581,-0.016981,-0.065453,0.058739,-0.006989,...,-0.056107,0.041549,0.000237,0.066626,0.042579,26.241991,1.965061,0.081824,4.293521,0.0
1,COBEN-WTABLET-DM-AD15,-0.051845,0.043946,0.011904,-0.010953,-0.025803,-0.017803,-0.065916,0.059028,-0.006370,...,0.041338,0.012307,0.000107,0.017672,0.010502,18.809973,1.891182,0.090616,4.502542,1.0
2,COBEN-WTABLET-DS-HC36,-0.053084,0.043164,0.011447,-0.010110,-0.025780,-0.016541,-0.065701,0.058689,-0.006781,...,-1.300261,0.010903,0.000142,0.018111,0.011886,21.863560,1.624943,0.144086,5.076851,0.0
3,COBEN-WTABLET-DV-AD04,-0.054663,0.041894,0.012110,-0.010013,-0.024074,-0.016389,-0.065150,0.055417,-0.007816,...,0.015836,0.006629,0.000076,0.008595,0.004086,16.379484,2.371073,0.099080,7.525870,1.0
4,COBEN-WTABLET-EH-HC35,-0.053226,0.043825,0.011746,-0.010794,-0.025438,-0.016439,-0.065685,0.058769,-0.006327,...,-0.056107,0.011419,-0.000003,0.018186,0.011020,11.250131,2.830892,0.111882,4.425100,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,HC-32#1,-0.053320,0.042233,0.012059,-0.010998,-0.025761,-0.017718,-0.065758,0.058969,-0.007252,...,-0.056107,0.154359,0.000451,0.209169,0.098028,24.558513,1.373170,0.134250,6.290672,0.0
71,HC-33#1,-0.053044,0.043273,0.011899,-0.010504,-0.025557,-0.016242,-0.065716,0.058804,-0.006392,...,-0.056107,0.018290,0.000127,0.026143,0.014093,15.589412,1.505466,0.077682,5.936073,0.0
72,HC-34#1,-0.052794,0.043555,0.012379,-0.010400,-0.025120,-0.016714,-0.065440,0.058418,-0.007094,...,-0.056107,0.013007,0.000044,0.018650,0.010024,19.916698,1.272528,0.150220,5.373593,0.0
73,HC-4#1,-0.052568,0.043872,0.011817,-0.011168,-0.025485,-0.017019,-0.065562,0.058993,-0.006118,...,-0.056107,0.003749,-0.000005,0.005173,0.002177,13.384290,1.909837,0.082338,4.804701,1.0


In [817]:
task_hf_compensated_freq_emb_lbl_ad_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_ad_hc.items()}
task_hf_compensated_temp_emb_lbl_ad_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_ad_hc.items()}
task_hf_compensated_comb_emb_lbl_ad_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_ad_hc.items()}

task_hf_compensated_freq_emb_lbl_lbd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_lbd_hc.items()}
task_hf_compensated_temp_emb_lbl_lbd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_lbd_hc.items()}
task_hf_compensated_comb_emb_lbl_lbd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_lbd_hc.items()}


task_hf_compensated_freq_emb_lbl_ad_lbl = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_ad_lbd.items()}
task_hf_compensated_temp_emb_lbl_ad_lbd = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_ad_lbd.items()}
task_hf_compensated_comb_emb_lbl_ad_lbd = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_ad_lbd.items()}

task_hf_compensated_freq_emb_lbl_pd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_pd_hc.items()}
task_hf_compensated_temp_emb_lbl_pd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_pd_hc.items()}
task_hf_compensated_comb_emb_lbl_pd_hc = {k: remove_leaking_labels(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_pd_hc.items()}

In [818]:
def leaking_sanity_test(df, label_string='diagnosis_x'):
    if label_string in df.columns:
        print(f"Leaking label '{label_string}' found in DataFrame columns.")
    else:
        print(f"No leaking label '{label_string}' found in DataFrame columns.")

In [782]:
{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_ad_hc.items()}
{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_ad_hc.items()}
{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_ad_hc.items()}

{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_lbd_hc.items()}
{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_lbd_hc.items()}
{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_lbd_hc.items()}


{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_freq_emb_lbl_ad_lbl.items()}
{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_temp_emb_lbl_ad_lbd.items()}
{k: leaking_sanity_test(v, label_string = 'diagnosis_x') for k,v in task_hf_compensated_comb_emb_lbl_ad_lbd.items()}

No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame columns.
No leaking label 'diagnosis_x' found in DataFrame column

{'1_1': None,
 '17_1': None,
 '16_1': None,
 '18_1': None,
 '9_1': None,
 '15_1': None,
 '19_1': None}

In [644]:
task_hf_compensated_comb_emb_lbl_lbd_hc.get("1_1")

,subject,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,w.cz.fnusa.1_1_slope of duration of pen stops,w.cz.fnusa.1_1_slope of horizontal velocity (on-surface),w.cz.fnusa.1_1_slope of pressure,w.cz.fnusa.1_1_slope of velocity (on-surface),w.cz.fnusa.1_1_slope of vertical velocity (on-surface),w.cz.fnusa.1_1_spiral precision index,w.cz.fnusa.1_1_tightness of spiral,w.cz.fnusa.1_1_variability of spiral width,w.cz.fnusa.1_1_zero-crossing rate of spiral,diagnosis_y
0,COBEN-WTABLET-AS-HCD05,0.069998,0.051461,0.026161,0.029638,0.002900,-0.033204,-0.038481,0.040194,0.013137,...,-0.036781,0.041549,0.000237,0.066626,0.042579,26.241991,1.965061,0.081824,4.293521,0.0
1,COBEN-WTABLET-DS-HC36,0.070257,0.052253,0.025544,0.029721,0.001458,-0.032596,-0.038438,0.040032,0.013936,...,-1.300261,0.010903,0.000142,0.018111,0.011886,21.863560,1.624943,0.144086,5.076851,0.0
2,COBEN-WTABLET-EH-HC35,0.070447,0.052164,0.025525,0.029518,0.002158,-0.032526,-0.037813,0.040152,0.014474,...,-0.036781,0.011419,-0.000003,0.018186,0.011020,11.250131,2.830892,0.111882,4.425100,0.0
3,COBEN-WTABLET-FG-HC38,0.068992,0.076354,0.024521,0.021995,-0.003542,-0.018538,-0.043351,0.021787,0.019038,...,-0.036781,0.000988,-0.000015,0.000974,0.000441,15.274915,2.324467,0.087784,5.158730,0.0
4,COBEN-WTABLET-HJ-HC21,0.069588,0.050415,0.025693,0.029191,0.002059,-0.033489,-0.037357,0.041113,0.014325,...,-0.036781,0.002055,0.000018,0.003600,0.001887,24.181988,2.697463,0.138062,3.830944,0.0
5,COBEN-WTABLET-HJ-HC29,0.070530,0.053146,0.025328,0.029738,0.001855,-0.032290,-0.038542,0.040080,0.013744,...,-0.036781,0.006889,-0.000008,0.011469,0.007232,11.495067,2.475328,0.082729,6.547866,0.0
6,COBEN-WTABLET-HS-HC37,0.070413,0.052078,0.025414,0.030180,0.001857,-0.032931,-0.038214,0.040429,0.013815,...,-0.036781,0.028932,0.000111,0.046567,0.029165,13.720159,2.021142,0.086121,5.497171,0.0
7,COBEN-WTABLET-HZ-HC13,0.070874,0.051366,0.025257,0.029971,0.001613,-0.033162,-0.037655,0.040345,0.014644,...,-0.036781,0.055995,0.000363,0.084254,0.051700,20.636313,2.190370,0.100018,8.396125,0.0
8,COBEN-WTABLET-IK-HC25,0.069278,0.052119,0.025852,0.030265,0.000870,-0.032181,-0.037740,0.040134,0.013661,...,-0.255542,-0.004840,0.000086,-0.007412,-0.004554,11.561671,-0.086989,0.129958,1.844140,0.0
9,COBEN-WTABLET-IK-HC26,0.070288,0.053917,0.024299,0.031992,-0.000355,-0.030294,-0.038207,0.039371,0.012000,...,-0.052617,0.076562,0.000297,0.102571,0.056834,13.780890,-1.359569,0.129958,6.327373,0.0


# OPTUNA + XGBoost

In [630]:

import os, json
import numpy as np
import pandas as pd
import optuna, xgboost as xgb
from typing import Dict, Tuple
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
    ConfusionMatrixDisplay,
    matthews_corrcoef
)

# def x_y_split(df): ...  # your splitter (must return X, y)
def _safe_feature_names(X, fallback_dim=None):
    if hasattr(X, "columns"):
        return list(X.columns)
    if hasattr(X, "feature_names_in_"):
        return list(X.feature_names_in_)
    if fallback_dim is None and hasattr(X, "shape"):
        fallback_dim = X.shape[1]
    return [f"f{i}" for i in range(int(fallback_dim or 0))]

def _plot_feature_importance(model, feature_names, out_dir: str):
    """Save multiple feature-importance views from XGBoost."""
    import matplotlib.pyplot as plt
    import numpy as np
    from collections import defaultdict
    os.makedirs(out_dir, exist_ok=True)

    # 1) sklearn API importances (gain-based)
    try:
        importances = getattr(model, "feature_importances_", None)
        if importances is not None and len(importances) == len(feature_names):
            order = np.argsort(importances)[::-1]
            top_idx = order[:50]  # limit to top-50 for readability
            plt.figure()
            plt.barh([feature_names[i] for i in top_idx][::-1], importances[top_idx][::-1])
            plt.title("XGB feature_importances_ (top-50)")
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "feature_importances_sklearn.png"), dpi=200, bbox_inches="tight")
            plt.close()
    except Exception:
        pass

    # 2) Booster importances (weight/gain/cover)
    try:
        booster = model.get_booster()
        for typ in ["weight", "gain", "cover", "total_gain", "total_cover"]:
            score_dict = booster.get_score(importance_type=typ)
            if not score_dict:
                continue
            # Map 'f0'.. to friendly names
            vals = []
            names = []
            for k, v in score_dict.items():
                if k.startswith("f"):
                    idx = int(k[1:])
                    if 0 <= idx < len(feature_names):
                        names.append(feature_names[idx])
                    else:
                        names.append(k)
                else:
                    names.append(k)
                vals.append(v)
            order = np.argsort(vals)[::-1]
            top = min(50, len(order))
            plt.figure()
            plt.barh([names[i] for i in order[:top]][::-1], np.array(vals)[order[:top]][::-1])
            plt.title(f"Booster feature importance — {typ} (top-{top})")
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"feature_importances_{typ}.png"), dpi=200, bbox_inches="tight")
            plt.close()
    except Exception:
        pass

def _save_shap_summaries(model, X_ref, feature_names, out_dir: str, max_display: int = 30, sample: int = 1000):
    """
    Compute SHAP (TreeExplainer) and save bar + beeswarm plots.
    Skips silently if shap not installed or backend issues arise.
    """
    try:
        import shap
        import numpy as np
        import matplotlib.pyplot as plt
    except Exception:
        return

    try:
        # Downsample for speed
        if hasattr(X_ref, "iloc"):
            X_use = X_ref.sample(min(sample, len(X_ref)), random_state=0)
            X_np = X_use.values
        else:
            X_np = X_ref
            if X_np.shape[0] > sample:
                rng = np.random.default_rng(0)
                idx = rng.choice(X_np.shape[0], size=sample, replace=False)
                X_np = X_np[idx]

        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_np)

        # Handle binary classification: shap returns array (n, p)
        if isinstance(shap_values, list) and len(shap_values) == 2:
            # pick positive class
            sv = shap_values[1]
        else:
            sv = shap_values

        # Bar plot (mean |SHAP|)
        try:
            plt.figure()
            shap.summary_plot(sv, X_np, feature_names=feature_names, plot_type="bar", show=False, max_display=max_display)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "shap_summary_bar.png"), dpi=200, bbox_inches="tight")
            plt.close()
        except Exception:
            pass

        # Beeswarm
        try:
            plt.figure()
            shap.summary_plot(sv, X_np, feature_names=feature_names, show=False, max_display=max_display)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "shap_summary_beeswarm.png"), dpi=200, bbox_inches="tight")
            plt.close()
        except Exception:
            pass
    except Exception:
        # Any SHAP error -> skip silently
        return

def _pick_device():
    try:
        _ = xgb.core.get_cuda_compute_capabilities()
        return {"tree_method": "hist", "device": "cuda"}
    except Exception:
        return {"tree_method": "hist", "device": "cpu"}

def _ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def _save_json(d: dict, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(d, f, indent=2, ensure_ascii=False)

def make_xgb_objective(X, y, scoring: str = "balanced_accuracy", n_splits: int = 5, random_state: int = 42):
    if scoring == "balanced_accuracy":
        from sklearn.metrics import balanced_accuracy_score as metric_fn
        needs_proba = False
        eval_metric = "logloss"
    elif scoring == "roc_auc":
        from sklearn.metrics import roc_auc_score as metric_fn
        needs_proba = True
        eval_metric = "auc"
    else:
        raise ValueError("scoring must be 'balanced_accuracy' or 'roc_auc'")

    folds = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    #folds = LeaveOneOut()
    device_kwargs = _pick_device()

    pos_ratio = float(np.mean(y))
    scale_pos_weight = ((1.0 - pos_ratio) / pos_ratio) if 0 < pos_ratio < 1 else 1.0

    def objective(trial: optuna.Trial) -> float:
        params = {
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 200, 1500),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
            "objective": "binary:logistic",
            "eval_metric": eval_metric,
            "n_jobs": 1,  # keep each trial single-threaded
            "scale_pos_weight": scale_pos_weight,
            **device_kwargs,
        }

        scores = []
        for tr_idx, va_idx in folds.split(X, y):
            if hasattr(X, "iloc"):
                X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
                y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
            else:
                X_tr, X_va = X[tr_idx], X[va_idx]
                y_tr, y_va = y[tr_idx], y[va_idx]

            model = xgb.XGBClassifier(**params)
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                #early_stopping_rounds=50,
                verbose=False,
            )

            if needs_proba:
                y_score = model.predict_proba(X_va)[:, 1]
                score = metric_fn(y_va, y_score)
            else:
                y_pred = model.predict(X_va)
                score = metric_fn(y_va, y_pred)
            scores.append(score)

            trial.report(np.mean(scores), len(scores))
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(scores))

    return objective

def _evaluate_and_save(name: str, model, X_test, y_test, out_dir: str, feature_names=None, X_ref_for_shap=None):
    """Compute metrics, plot ROC + confusion matrix, and save everything."""
    _ensure_dir(out_dir)

    # Predictions
    y_proba = None
    try:
        y_proba = model.predict_proba(X_test)[:, 1]
    except Exception:
        pass
    y_pred = model.predict(X_test)

    # Metrics
    metrics = {
        "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
        "mcc": None,
        "roc_auc": None,
        "sensitivity": None,   # <-- NEW
        "specificity": None,   # <-- NEW
        "classification_report": None,
    }
    # MCC
    try:
        metrics["mcc"] = float(matthews_corrcoef(y_test, y_pred))
    except Exception:
        pass
    # ROC-AUC if both classes present and proba available
    try:
        if y_proba is not None and len(np.unique(y_test)) == 2:
            metrics["roc_auc"] = float(roc_auc_score(y_test, y_proba))
        else:
            metrics["roc_auc"] = None
    except Exception:
        metrics["roc_auc"] = None
    # --- Sensitivity & Specificity (labels fixed to [0,1]) ---
    try:
        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        sens_den = tp + fn
        spec_den = tn + fp
        metrics["sensitivity"] = float(tp / sens_den) if sens_den > 0 else None
        metrics["specificity"] = float(tn / spec_den) if spec_den > 0 else None
    except Exception:
        pass


    # Classification report (as dict)
    try:
        metrics["classification_report"] = classification_report(y_test, y_pred, output_dict=True)
    except Exception:
        metrics["classification_report"] = None

    # Confusion matrix plot
    try:
        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(cm)
        import matplotlib.pyplot as plt
        plt.figure()
        disp.plot(values_format="d")
        plt.title(f"Confusion Matrix — {name}")
        plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=200, bbox_inches="tight")
        plt.close()
    except Exception:
        pass

    # ROC curve plot
    try:
        if y_proba is not None and len(np.unique(y_test)) == 2:
            import matplotlib.pyplot as plt
            plt.figure()
            RocCurveDisplay.from_predictions(y_test, y_proba)
            plt.title(f"ROC Curve — {name}")
            plt.savefig(os.path.join(out_dir, "roc_curve.png"), dpi=200, bbox_inches="tight")
            plt.close()
    except Exception:
        pass

    # Feature names
    if feature_names is None:
        feature_names = _safe_feature_names(X_test)

    # Feature importance plots
    try:
        _plot_feature_importance(model, feature_names, out_dir)
    except Exception:
        pass

    # SHAP plots (optional)
    try:
        if X_ref_for_shap is None:
            X_ref_for_shap = X_test
        _save_shap_summaries(model, X_ref_for_shap, feature_names, out_dir)
    except Exception:
        pass

    # Save metrics JSON
    _save_json(metrics, os.path.join(out_dir, "metrics.json"))
    return metrics

def optimize_many(
    dfs: Dict[str, pd.DataFrame],
    x_y_split_fn,
    scoring: str = "balanced_accuracy",
    n_trials: int = 500,
    n_splits: int = 5,
    random_state: int = 42,
    n_jobs_trials: int = -1,
    output_dir: str = "xgb_results",
    test_size: float = 0.20,
    split_param: str = "emb_",
) -> Tuple[pd.DataFrame, Dict[str, xgb.XGBClassifier], Dict[str, optuna.Study]]:
    """
    Runs Optuna per dataset, evaluates on a held-out test set,
    and saves plots + metrics in output_dir/<dataset_name>/
    """
    results = []
    best_models: Dict[str, xgb.XGBClassifier] = {}
    studies: Dict[str, optuna.Study] = {}

    _ensure_dir(output_dir)

    for name, df in dfs.items():
        X, y = x_y_split_fn(df, split_param=split_param)

        # To numpy for fast indexing, but keep DataFrame support
        X_arr = X.values if hasattr(X, "values") else X
        y_arr = y.values if hasattr(y, "values") else y

        feature_names = _safe_feature_names(X)
        # Keep names before converting to numpy

        # Hold-out test split (kept untouched for final evaluation)
        X_train, X_test, y_train, y_test = train_test_split(
            X_arr, y_arr, test_size=test_size, random_state=random_state, stratify=y_arr
        )

        # Build objective on TRAIN ONLY
        objective = make_xgb_objective(X_train, y_train, scoring=scoring, n_splits=n_splits, random_state=random_state)

        study = optuna.create_study(
            study_name=f"{name}_{scoring}",
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=random_state),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=10),
        )
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True, n_jobs=n_jobs_trials)
        studies[name] = study

        # Train final model on TRAIN using best params (with small VA split for early stopping)
        device_kwargs = _pick_device()
        pos_ratio = float(np.mean(y_train))
        spw = ((1.0 - pos_ratio) / pos_ratio) if 0 < pos_ratio < 1 else 1.0

        best_params = {
            **study.best_params,
            "objective": "binary:logistic",
            "eval_metric": "auc" if scoring == "roc_auc" else "logloss",
            "n_jobs": 0,
            "scale_pos_weight": spw,
            **device_kwargs,
        }
        final_model = xgb.XGBClassifier(**best_params)

        # Early stopping split inside TRAIN
        X_tr, X_va, y_tr, y_va = train_test_split(
            X_train, y_train, test_size=0.15, random_state=random_state, stratify=y_train
        )
        final_model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            #early_stopping_rounds=50,
            verbose=False,
        )

        # Save per-dataset artifacts
        ds_out = os.path.join(output_dir, name)
        _ensure_dir(ds_out)

        # Evaluate on TEST and save plots/metrics
        metrics = _evaluate_and_save(
            name, final_model, X_test, y_test, ds_out,
            feature_names=feature_names,
            X_ref_for_shap=X_test
        )

        # Save best params + study summary
        _save_json({"best_params": study.best_params, "best_value": study.best_value}, os.path.join(ds_out, "best.json"))

        # Optional: save model
        try:
            import joblib
            joblib.dump(final_model, os.path.join(ds_out, "model.joblib"))
        except Exception:
            pass

        best_models[name] = final_model
        results.append({
            "dataset": name,
            "cv_best_value": study.best_value,
            "test_balanced_accuracy": metrics.get("balanced_accuracy"),
            "test_roc_auc": metrics.get("roc_auc"),
            "n_trials": len(study.trials),
        })

    summary_df = pd.DataFrame(results).sort_values(by="test_balanced_accuracy", ascending=False).reset_index(drop=True)
    # Save a global summary too
    summary_df.to_csv(os.path.join(output_dir, "summary.csv"), index=False)
    return summary_df, best_models, studies


# MCI_LBD vs HC -> OPTUNA + XGB

# From this one

In [633]:
task_hf_compensated_comb_emb_lbl_lbd_hc.get("1_1")

,subject,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,w.cz.fnusa.1_1_slope of duration of pen stops,w.cz.fnusa.1_1_slope of horizontal velocity (on-surface),w.cz.fnusa.1_1_slope of pressure,w.cz.fnusa.1_1_slope of velocity (on-surface),w.cz.fnusa.1_1_slope of vertical velocity (on-surface),w.cz.fnusa.1_1_spiral precision index,w.cz.fnusa.1_1_tightness of spiral,w.cz.fnusa.1_1_variability of spiral width,w.cz.fnusa.1_1_zero-crossing rate of spiral,diagnosis_y
0,COBEN-WTABLET-AS-HCD05,0.069998,0.051461,0.026161,0.029638,0.002900,-0.033204,-0.038481,0.040194,0.013137,...,-0.036781,0.041549,0.000237,0.066626,0.042579,26.241991,1.965061,0.081824,4.293521,0.0
1,COBEN-WTABLET-DS-HC36,0.070257,0.052253,0.025544,0.029721,0.001458,-0.032596,-0.038438,0.040032,0.013936,...,-1.300261,0.010903,0.000142,0.018111,0.011886,21.863560,1.624943,0.144086,5.076851,0.0
2,COBEN-WTABLET-EH-HC35,0.070447,0.052164,0.025525,0.029518,0.002158,-0.032526,-0.037813,0.040152,0.014474,...,-0.036781,0.011419,-0.000003,0.018186,0.011020,11.250131,2.830892,0.111882,4.425100,0.0
3,COBEN-WTABLET-FG-HC38,0.068992,0.076354,0.024521,0.021995,-0.003542,-0.018538,-0.043351,0.021787,0.019038,...,-0.036781,0.000988,-0.000015,0.000974,0.000441,15.274915,2.324467,0.087784,5.158730,0.0
4,COBEN-WTABLET-HJ-HC21,0.069588,0.050415,0.025693,0.029191,0.002059,-0.033489,-0.037357,0.041113,0.014325,...,-0.036781,0.002055,0.000018,0.003600,0.001887,24.181988,2.697463,0.138062,3.830944,0.0
5,COBEN-WTABLET-HJ-HC29,0.070530,0.053146,0.025328,0.029738,0.001855,-0.032290,-0.038542,0.040080,0.013744,...,-0.036781,0.006889,-0.000008,0.011469,0.007232,11.495067,2.475328,0.082729,6.547866,0.0
6,COBEN-WTABLET-HS-HC37,0.070413,0.052078,0.025414,0.030180,0.001857,-0.032931,-0.038214,0.040429,0.013815,...,-0.036781,0.028932,0.000111,0.046567,0.029165,13.720159,2.021142,0.086121,5.497171,0.0
7,COBEN-WTABLET-HZ-HC13,0.070874,0.051366,0.025257,0.029971,0.001613,-0.033162,-0.037655,0.040345,0.014644,...,-0.036781,0.055995,0.000363,0.084254,0.051700,20.636313,2.190370,0.100018,8.396125,0.0
8,COBEN-WTABLET-IK-HC25,0.069278,0.052119,0.025852,0.030265,0.000870,-0.032181,-0.037740,0.040134,0.013661,...,-0.255542,-0.004840,0.000086,-0.007412,-0.004554,11.561671,-0.086989,0.129958,1.844140,0.0
9,COBEN-WTABLET-IK-HC26,0.070288,0.053917,0.024299,0.031992,-0.000355,-0.030294,-0.038207,0.039371,0.012000,...,-0.052617,0.076562,0.000297,0.102571,0.056834,13.780890,-1.359569,0.129958,6.327373,0.0


In [647]:
task_hf_compensated_comb_emb_lbl_lbd_hc.get("1_1")

,subject,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,w.cz.fnusa.1_1_slope of duration of pen stops,w.cz.fnusa.1_1_slope of horizontal velocity (on-surface),w.cz.fnusa.1_1_slope of pressure,w.cz.fnusa.1_1_slope of velocity (on-surface),w.cz.fnusa.1_1_slope of vertical velocity (on-surface),w.cz.fnusa.1_1_spiral precision index,w.cz.fnusa.1_1_tightness of spiral,w.cz.fnusa.1_1_variability of spiral width,w.cz.fnusa.1_1_zero-crossing rate of spiral,diagnosis_y
0,COBEN-WTABLET-AS-HCD05,0.069998,0.051461,0.026161,0.029638,0.002900,-0.033204,-0.038481,0.040194,0.013137,...,-0.036781,0.041549,0.000237,0.066626,0.042579,26.241991,1.965061,0.081824,4.293521,0.0
1,COBEN-WTABLET-DS-HC36,0.070257,0.052253,0.025544,0.029721,0.001458,-0.032596,-0.038438,0.040032,0.013936,...,-1.300261,0.010903,0.000142,0.018111,0.011886,21.863560,1.624943,0.144086,5.076851,0.0
2,COBEN-WTABLET-EH-HC35,0.070447,0.052164,0.025525,0.029518,0.002158,-0.032526,-0.037813,0.040152,0.014474,...,-0.036781,0.011419,-0.000003,0.018186,0.011020,11.250131,2.830892,0.111882,4.425100,0.0
3,COBEN-WTABLET-FG-HC38,0.068992,0.076354,0.024521,0.021995,-0.003542,-0.018538,-0.043351,0.021787,0.019038,...,-0.036781,0.000988,-0.000015,0.000974,0.000441,15.274915,2.324467,0.087784,5.158730,0.0
4,COBEN-WTABLET-HJ-HC21,0.069588,0.050415,0.025693,0.029191,0.002059,-0.033489,-0.037357,0.041113,0.014325,...,-0.036781,0.002055,0.000018,0.003600,0.001887,24.181988,2.697463,0.138062,3.830944,0.0
5,COBEN-WTABLET-HJ-HC29,0.070530,0.053146,0.025328,0.029738,0.001855,-0.032290,-0.038542,0.040080,0.013744,...,-0.036781,0.006889,-0.000008,0.011469,0.007232,11.495067,2.475328,0.082729,6.547866,0.0
6,COBEN-WTABLET-HS-HC37,0.070413,0.052078,0.025414,0.030180,0.001857,-0.032931,-0.038214,0.040429,0.013815,...,-0.036781,0.028932,0.000111,0.046567,0.029165,13.720159,2.021142,0.086121,5.497171,0.0
7,COBEN-WTABLET-HZ-HC13,0.070874,0.051366,0.025257,0.029971,0.001613,-0.033162,-0.037655,0.040345,0.014644,...,-0.036781,0.055995,0.000363,0.084254,0.051700,20.636313,2.190370,0.100018,8.396125,0.0
8,COBEN-WTABLET-IK-HC25,0.069278,0.052119,0.025852,0.030265,0.000870,-0.032181,-0.037740,0.040134,0.013661,...,-0.255542,-0.004840,0.000086,-0.007412,-0.004554,11.561671,-0.086989,0.129958,1.844140,0.0
9,COBEN-WTABLET-IK-HC26,0.070288,0.053917,0.024299,0.031992,-0.000355,-0.030294,-0.038207,0.039371,0.012000,...,-0.052617,0.076562,0.000297,0.102571,0.056834,13.780890,-1.359569,0.129958,6.327373,0.0


In [819]:
#TODO: Contains just zeroes in diagnosis, need to fix 
datasets = task_hf_comb_emb_lbl_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_lbd_hc/xgb_results_handcrafted_combined_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)


[I 2025-11-24 17:12:07,246] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 2. Best value: 0.801111:   0%|          | 1/200 [00:07<23:42,  7.15s/it]

[I 2025-11-24 17:12:14,383] Trial 2 finished with value: 0.8011111111111111 and parameters: {'max_depth': 6, 'learning_rate': 0.09978957740540101, 'n_estimators': 570, 'subsample': 0.8685789905111967, 'colsample_bytree': 0.7778304246336839, 'min_child_weight': 8, 'gamma': 1.3755996894611928, 'reg_alpha': 1.0399202701145535e-05, 'reg_lambda': 2.4500997178294732e-08, 'max_delta_step': 5}. Best is trial 2 with value: 0.8011111111111111.


Best trial: 3. Best value: 1:   1%|          | 2/200 [00:09<13:19,  4.04s/it]       

[I 2025-11-24 17:12:16,245] Trial 3 finished with value: 1.0 and parameters: {'max_depth': 9, 'learning_rate': 0.017536268544503863, 'n_estimators': 720, 'subsample': 0.67143725001995, 'colsample_bytree': 0.5698271708362341, 'min_child_weight': 5, 'gamma': 1.820638241283895, 'reg_alpha': 1.855210767276572e-08, 'reg_lambda': 0.0001883395209635505, 'max_delta_step': 4}. Best is trial 3 with value: 1.0.


Best trial: 3. Best value: 1:   2%|▏         | 3/200 [00:09<07:58,  2.43s/it]

...

Best trial: 3. Best value: 1:   4%|▍         | 8/200 [00:18<07:17,  2.28s/it]

[I 2025-11-24 17:12:26,152] Trial 6 finished with value: 1.0 and parameters: {'max_depth': 5, 'learning_rate': 0.021181479088540286, 'n_estimators': 840, 'subsample': 0.740875203075567, 'colsample_bytree': 0.7389564345910318, 'min_child_weight': 4, 'gamma': 0.1158297777788786, 'reg_alpha': 3.280262081046563e-05, 'reg_lambda': 1.7946276645190308e-06, 'max_delta_step': 8}. Best is trial 3 with value: 1.0.


Best trial: 3. Best value: 1:   4%|▍         | 9/200 [00:20<07:08,  2.25s/it]

[I 2025-11-24 17:12:27,451] Trial 8 finished with value: 0.5 and parameters: {'max_depth': 3, 'learning_rate': 0.06969033786973476, 'n_estimators': 1083, 'subsample': 0.6711904736098027, 'colsample_bytree': 0.7512968824147984, 'min_child_weight': 8, 'gamma': 2.7865440006438797, 'reg_alpha': 0.10468089714314044, 'reg_lambda': 0.30342763157083225, 'max_delta_step': 10}. Best is trial 3 with value: 1.0.


KeyboardInterrupt: 

In [377]:
task_hf_compensated_comb_emb_lbl_lbd_hc.get("1_1")

,label,subject,file_path,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,...,w.cz.fnusa.1_1_slope of duration of pen stops,w.cz.fnusa.1_1_slope of horizontal velocity (on-surface),w.cz.fnusa.1_1_slope of pressure,w.cz.fnusa.1_1_slope of velocity (on-surface),w.cz.fnusa.1_1_slope of vertical velocity (on-surface),w.cz.fnusa.1_1_spiral precision index,w.cz.fnusa.1_1_tightness of spiral,w.cz.fnusa.1_1_variability of spiral width,w.cz.fnusa.1_1_zero-crossing rate of spiral,diagnosis_y
0,1,COBEN-WTABLET-AS-HCD05,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.069998,0.051461,0.026161,0.029638,0.002900,-0.033204,-0.038481,...,-0.036781,0.041549,0.000237,0.066626,0.042579,26.241991,1.965061,0.081824,4.293521,0.0
1,1,COBEN-WTABLET-DS-HC36,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.070257,0.052253,0.025544,0.029721,0.001458,-0.032596,-0.038438,...,-1.300261,0.010903,0.000142,0.018111,0.011886,21.863560,1.624943,0.144086,5.076851,0.0
2,1,COBEN-WTABLET-EH-HC35,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.070447,0.052164,0.025525,0.029518,0.002158,-0.032526,-0.037813,...,-0.036781,0.011419,-0.000003,0.018186,0.011020,11.250131,2.830892,0.111882,4.425100,0.0
3,1,COBEN-WTABLET-FG-HC38,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.068992,0.076354,0.024521,0.021995,-0.003542,-0.018538,-0.043351,...,-0.036781,0.000988,-0.000015,0.000974,0.000441,15.274915,2.324467,0.087784,5.158730,0.0
4,1,COBEN-WTABLET-HJ-HC21,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.069588,0.050415,0.025693,0.029191,0.002059,-0.033489,-0.037357,...,-0.036781,0.002055,0.000018,0.003600,0.001887,24.181988,2.697463,0.138062,3.830944,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,1,pre-LBD-56#1,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.070023,0.056273,0.023943,0.030077,-0.000505,-0.031746,-0.039750,...,-0.060143,0.005880,0.000096,0.009366,0.005375,17.459382,2.186787,0.137749,7.024557,1.0
80,1,pre-LBD-57#1,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.069374,0.055039,0.025324,0.030729,0.001077,-0.031234,-0.038828,...,0.007518,0.002588,0.000010,0.004087,0.002246,15.178891,2.670635,0.137702,7.082593,1.0
81,1,pre-LBD-57#2,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.084256,0.052775,0.028871,0.023229,0.002501,-0.010873,-0.041387,...,-0.018527,0.003621,-0.000023,0.005623,0.003155,23.956094,2.949314,0.171941,5.133268,1.0
82,1,pre-LBD-58#1,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.070419,0.055387,0.024555,0.029948,0.000351,-0.032082,-0.039027,...,-0.036781,0.019840,0.000091,0.024970,0.012266,17.709650,2.236651,0.116427,3.826531,1.0


In [ ]:
task_hf_compensated_comb_emb_lbl_lbd_hc.get('1_1')

In [820]:

datasets = task_hf_compensated_temp_emb_lbl_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_lbd_hc/xgb_results_handcrafted_temporal_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)


[I 2025-11-24 17:12:34,535] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 3. Best value: 0.595278:   0%|          | 1/200 [00:03<12:45,  3.85s/it]

[I 2025-11-24 17:12:38,366] Trial 0 finished with value: 0.46944444444444444 and parameters: {'max_depth': 4, 'learning_rate': 0.07953331647321166, 'n_estimators': 254, 'subsample': 0.9020582975357148, 'colsample_bytree': 0.9391293282751374, 'min_child_weight': 8, 'gamma': 3.0442368118543985, 'reg_alpha': 1.3102562614332123e-08, 'reg_lambda': 4.668706995433403e-05, 'max_delta_step': 3}. Best is trial 0 with value: 0.46944444444444444.
[I 2025-11-24 17:12:38,379] Trial 3 finished with value: 0.5952777777777778 and parameters: {'max_depth': 3, 'learning_rate': 0.019448771527328275, 'n_estimators': 254, 'subsample': 0.9789314911233455, 'colsample_bytree': 0.5020444834388402, 'min_child_weight': 8, 'gamma': 3.7629099921293707, 'reg_alpha': 6.253996057262623e-06, 'reg_lambda': 0.09502997312810064, 'max_delta_step': 0}. Best is trial 3 with value: 0.5952777777777778.


Best trial: 3. Best value: 0.595278:   2%|▏         | 3/200 [00:07<07:11,  2.19s/it]

[I 2025-11-24 17:12:41,589] Trial 4 finished with value: 0.5 and parameters: {'max_depth': 3, 'learning_rate': 0.013343951551124727, 'n_estimators': 261, 'subsample': 0.6207063563517625, 'colsample_bytree': 0.5217193609411814, 'min_child_weight': 7, 'gamma': 4.112666823498496, 'reg_alpha': 3.91104548358634e-06, 'reg_lambda': 4.5269039407268305e-05, 'max_delta_step': 1}. Best is trial 3 with value: 0.5952777777777778.


Best trial: 3. Best value: 0.595278:   2%|▏         | 4/200 [00:09<07:09,  2.19s/it]

...

Best trial: 3. Best value: 0.595278:   2%|▎         | 5/200 [00:09<05:29,  1.69s/it]

[I 2025-11-24 17:12:44,465] Trial 2 finished with value: 0.49333333333333335 and parameters: {'max_depth': 6, 'learning_rate': 0.048480096861828526, 'n_estimators': 890, 'subsample': 0.8367366796162705, 'colsample_bytree': 0.6949632216958934, 'min_child_weight': 2, 'gamma': 3.6295319281211134, 'reg_alpha': 0.11911706840934416, 'reg_lambda': 2.7268021474070828e-08, 'max_delta_step': 2}. Best is trial 3 with value: 0.5952777777777778.


Best trial: 3. Best value: 0.595278:   3%|▎         | 6/200 [00:11<05:59,  1.86s/it]

[I 2025-11-24 17:12:45,663] Trial 5 finished with value: 0.586111111111111 and parameters: {'max_depth': 10, 'learning_rate': 0.01776402101417324, 'n_estimators': 898, 'subsample': 0.8603691895707861, 'colsample_bytree': 0.9220321334912123, 'min_child_weight': 7, 'gamma': 1.0680758550891212, 'reg_alpha': 1.6229255665808784e-06, 'reg_lambda': 1.1613145739984241e-08, 'max_delta_step': 9}. Best is trial 3 with value: 0.5952777777777778.


KeyboardInterrupt: 

In [ ]:
task_hf_compensated_temp_emb_lbl_lbd_hc.get('18_1').head(15)

In [648]:

datasets = task_hf_compensated_freq_emb_lbl_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_lbd_hc/xgb_results_handcrafted_frerq_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)
## MCI-AD vs HC -> OPTUNA + XGB

[I 2025-11-24 16:37:13,214] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 2. Best value: 0.474722:   0%|          | 1/200 [00:06<21:11,  6.39s/it]

[I 2025-11-24 16:37:19,592] Trial 2 finished with value: 0.47472222222222216 and parameters: {'max_depth': 7, 'learning_rate': 0.08918743355583306, 'n_estimators': 555, 'subsample': 0.9381926388560211, 'colsample_bytree': 0.8825893089303807, 'min_child_weight': 4, 'gamma': 2.5734321218274934, 'reg_alpha': 1.4624430243342832e-08, 'reg_lambda': 0.3443920887278927, 'max_delta_step': 10}. Best is trial 2 with value: 0.47472222222222216.


Best trial: 1. Best value: 0.5:   1%|          | 2/200 [00:06<09:42,  2.94s/it]     

[I 2025-11-24 16:37:20,118] Trial 1 finished with value: 0.5 and parameters: {'max_depth': 7, 'learning_rate': 0.2163017258969618, 'n_estimators': 685, 'subsample': 0.6518270683639176, 'colsample_bytree': 0.8184059953089883, 'min_child_weight': 9, 'gamma': 4.986705984818089, 'reg_alpha': 2.900481706326526e-08, 'reg_lambda': 3.6544855862789514e-07, 'max_delta_step': 7}. Best is trial 1 with value: 0.5.


Best trial: 1. Best value: 0.5:   2%|▏         | 3/200 [00:10<09:53,  3.01s/it]

...

Best trial: 4. Best value: 0.544444:   3%|▎         | 6/200 [00:18<08:20,  2.58s/it]

[I 2025-11-24 16:37:31,354] Trial 4 finished with value: 0.5444444444444445 and parameters: {'max_depth': 9, 'learning_rate': 0.011482723971244526, 'n_estimators': 1201, 'subsample': 0.6850435222081088, 'colsample_bytree': 0.866807690050988, 'min_child_weight': 5, 'gamma': 1.8625866121591317, 'reg_alpha': 0.0072381619592875104, 'reg_lambda': 1.183362877312675, 'max_delta_step': 2}. Best is trial 4 with value: 0.5444444444444445.


Best trial: 4. Best value: 0.544444:   4%|▎         | 7/200 [00:18<08:38,  2.69s/it]

[I 2025-11-24 16:37:32,025] Trial 6 finished with value: 0.5083333333333334 and parameters: {'max_depth': 4, 'learning_rate': 0.038080575794627224, 'n_estimators': 1049, 'subsample': 0.5459597505810174, 'colsample_bytree': 0.5162213539114009, 'min_child_weight': 4, 'gamma': 0.2766749374451605, 'reg_alpha': 1.8768877018387091, 'reg_lambda': 0.007235697057271849, 'max_delta_step': 0}. Best is trial 4 with value: 0.5444444444444445.


KeyboardInterrupt: 

In [649]:
datasets = task_freq_dfs_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_lbd_hc/xgb_results_freq_ad_hc_extended",
    split_param="emb_",
)
print(summary)


[I 2025-11-24 16:37:39,590] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 1. Best value: 0.526944:   0%|          | 1/200 [00:03<11:48,  3.56s/it]

[I 2025-11-24 16:37:43,140] Trial 1 finished with value: 0.5269444444444444 and parameters: {'max_depth': 9, 'learning_rate': 0.29444559859276503, 'n_estimators': 359, 'subsample': 0.5636570594182599, 'colsample_bytree': 0.7295117969218559, 'min_child_weight': 5, 'gamma': 4.255091913694496, 'reg_alpha': 6.491961819571967e-08, 'reg_lambda': 3.62442388324769e-06, 'max_delta_step': 1}. Best is trial 1 with value: 0.5269444444444444.


Best trial: 1. Best value: 0.526944:   1%|          | 2/200 [00:08<15:18,  4.64s/it]

[I 2025-11-24 16:37:48,538] Trial 2 finished with value: 0.4522222222222222 and parameters: {'max_depth': 10, 'learning_rate': 0.14562286423554985, 'n_estimators': 981, 'subsample': 0.9967709686107356, 'colsample_bytree': 0.6468197818710772, 'min_child_weight': 6, 'gamma': 0.9141091862010214, 'reg_alpha': 0.15082412936852116, 'reg_lambda': 0.0013477481281778236, 'max_delta_step': 5}. Best is trial 1 with value: 0.5269444444444444.


Best trial: 1. Best value: 0.526944:   2%|▏         | 3/200 [00:10<09:51,  3.00s/it]

...

Best trial: 1. Best value: 0.526944:   4%|▍         | 8/200 [00:19<07:04,  2.21s/it]

[I 2025-11-24 16:37:59,467] Trial 7 finished with value: 0.5 and parameters: {'max_depth': 5, 'learning_rate': 0.019959938943585286, 'n_estimators': 816, 'subsample': 0.5928830432474643, 'colsample_bytree': 0.7347173880540778, 'min_child_weight': 8, 'gamma': 2.4135370787498722, 'reg_alpha': 0.0003156594852573295, 'reg_lambda': 1.4310401754843894e-06, 'max_delta_step': 1}. Best is trial 1 with value: 0.5269444444444444.


Best trial: 8. Best value: 0.551389:   5%|▌         | 10/200 [00:23<07:21,  2.33s/it]

[I 2025-11-24 16:38:02,815] Trial 8 finished with value: 0.5513888888888889 and parameters: {'max_depth': 3, 'learning_rate': 0.26132044559215045, 'n_estimators': 1446, 'subsample': 0.7056733261188997, 'colsample_bytree': 0.7036009148676936, 'min_child_weight': 7, 'gamma': 2.1049304919436467, 'reg_alpha': 1.0630221540454783, 'reg_lambda': 1.065332501061687e-05, 'max_delta_step': 8}. Best is trial 8 with value: 0.5513888888888889.
[I 2025-11-24 16:38:02,831] Trial 9 finished with value: 0.5 and parameters: {'max_depth': 7, 'learning_rate': 0.0840712181393472, 'n_estimators': 1433, 'subsample': 0.5339643980347284, 'colsample_bytree': 0.7433547210703702, 'min_child_weight': 9, 'gamma': 2.7242109363504574, 'reg_alpha': 7.490020327279355e-05, 'reg_lambda': 0.02374825361583344, 'max_delta_step': 1}. Best is trial 8 with value: 0.5513888888888889.


KeyboardInterrupt: 

In [382]:
task_freq_dfs_lbd_hc.get("1_1")

,label,subject,file_path,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,...,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383,diagnosis
0,1,COBEN-WTABLET-AS-HCD05,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,-0.053105,0.043560,0.012405,-0.010514,-0.025581,-0.016981,-0.065453,...,-0.088822,-0.022320,-0.026403,-0.013258,0.028018,-0.070962,0.068714,0.030881,0.056124,0.0
1,1,COBEN-WTABLET-DS-HC36,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,-0.053084,0.043164,0.011447,-0.010110,-0.025780,-0.016541,-0.065701,...,-0.089358,-0.021474,-0.026835,-0.013402,0.029021,-0.070640,0.069674,0.030071,0.056516,0.0
2,1,COBEN-WTABLET-EH-HC35,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,-0.053226,0.043825,0.011746,-0.010794,-0.025438,-0.016439,-0.065685,...,-0.089707,-0.021616,-0.027196,-0.013121,0.028772,-0.070659,0.068586,0.030430,0.057629,0.0
3,1,COBEN-WTABLET-FG-HC38,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,-0.055040,0.039800,0.012924,-0.009970,-0.022507,-0.016689,-0.065373,...,-0.086872,-0.021779,-0.027846,-0.015642,0.031131,-0.069999,0.063484,0.033880,0.054142,0.0
4,1,COBEN-WTABLET-HJ-HC21,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,-0.052350,0.043245,0.011101,-0.011332,-0.026283,-0.016835,-0.066375,...,-0.089052,-0.021148,-0.026934,-0.013177,0.029222,-0.071786,0.068372,0.029911,0.057885,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,1,pre-LBD-56#1,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,-0.052041,0.043842,0.011981,-0.010804,-0.025135,-0.017819,-0.065613,...,-0.088472,-0.021973,-0.026428,-0.012492,0.027609,-0.070973,0.069329,0.030555,0.055443,1.0
80,1,pre-LBD-57#1,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,-0.052870,0.043794,0.012545,-0.010356,-0.024927,-0.016839,-0.065123,...,-0.089317,-0.022107,-0.026710,-0.013081,0.027799,-0.070441,0.068575,0.031124,0.056201,1.0
81,1,pre-LBD-57#2,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,-0.053594,0.042589,0.012181,-0.010256,-0.024861,-0.017241,-0.065238,...,-0.088494,-0.022282,-0.027285,-0.013833,0.028558,-0.070117,0.067527,0.031199,0.055633,1.0
82,1,pre-LBD-58#1,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,-0.053200,0.043859,0.012134,-0.010375,-0.025091,-0.016805,-0.065880,...,-0.089681,-0.021463,-0.026823,-0.013037,0.028441,-0.070371,0.068925,0.030711,0.056595,1.0


In [650]:

# Temporal embedings
datasets = task_temp_dfs_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    #    scoring="balanced_accuracy",     # or "roc_auc"
    scoring="roc_auc",      
    n_trials=200,
    n_splits=5,
    output_dir="xgb_lbd_hc/xgb_results_temporal_extended",
    split_param="emb_",
)
print(summary)


[I 2025-11-24 16:38:20,036] A new study created in memory with name: 1_1_roc_auc
Best trial: 3. Best value: 0.583333:   0%|          | 1/200 [00:07<26:18,  7.93s/it]

[I 2025-11-24 16:38:27,959] Trial 3 finished with value: 0.5833333333333333 and parameters: {'max_depth': 5, 'learning_rate': 0.10926435232033856, 'n_estimators': 751, 'subsample': 0.8115657839319798, 'colsample_bytree': 0.6515874744703625, 'min_child_weight': 3, 'gamma': 4.111536847656631, 'reg_alpha': 0.000663868032792069, 'reg_lambda': 9.192344899920727e-08, 'max_delta_step': 5}. Best is trial 3 with value: 0.5833333333333333.


Best trial: 1. Best value: 0.671528:   1%|          | 2/200 [00:09<13:07,  3.98s/it]

[I 2025-11-24 16:38:29,161] Trial 1 finished with value: 0.6715277777777777 and parameters: {'max_depth': 8, 'learning_rate': 0.18983505947745413, 'n_estimators': 915, 'subsample': 0.810432250194078, 'colsample_bytree': 0.9203072641581835, 'min_child_weight': 7, 'gamma': 0.32502670459482796, 'reg_alpha': 0.03185374767581448, 'reg_lambda': 1.4536027737383534e-07, 'max_delta_step': 8}. Best is trial 1 with value: 0.6715277777777777.


Best trial: 1. Best value: 0.671528:   2%|▏         | 3/200 [00:14<15:39,  4.77s/it]

...

Best trial: 1. Best value: 0.671528:   4%|▍         | 9/200 [00:27<06:59,  2.20s/it]

[I 2025-11-24 16:38:47,880] Trial 6 finished with value: 0.5438888888888889 and parameters: {'max_depth': 5, 'learning_rate': 0.014232085851592797, 'n_estimators': 828, 'subsample': 0.8764893068425913, 'colsample_bytree': 0.6749088732689237, 'min_child_weight': 3, 'gamma': 0.16785215538729303, 'reg_alpha': 0.004259353159820383, 'reg_lambda': 0.03197588729313346, 'max_delta_step': 2}. Best is trial 1 with value: 0.6715277777777777.


Best trial: 1. Best value: 0.671528:   5%|▌         | 10/200 [00:30<09:37,  3.04s/it]


[I 2025-11-24 16:38:50,438] Trial 9 finished with value: 0.5 and parameters: {'max_depth': 4, 'learning_rate': 0.06771949020483825, 'n_estimators': 943, 'subsample': 0.562529492989885, 'colsample_bytree': 0.6326353371149827, 'min_child_weight': 10, 'gamma': 0.47862553319887147, 'reg_alpha': 0.00012922934218069057, 'reg_lambda': 1.8070502116389227, 'max_delta_step': 0}. Best is trial 1 with value: 0.6715277777777777.


KeyboardInterrupt: 

In [821]:
#TODO: Contains 2s in diagnosis
# Combined embedings
datasets = task_comb_dfs_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="roc_auc",     # or "roc_auc"
    n_trials=200,
    n_splits=5,
    output_dir="xgb_lbd_hc/xgb_results_combined_extended",
    split_param="emb_",
)
print(summary)



[I 2025-11-24 17:13:08,930] A new study created in memory with name: 1_1_roc_auc
Best trial: 3. Best value: 0.617222:   0%|          | 1/200 [00:07<24:26,  7.37s/it]

[I 2025-11-24 17:13:16,288] Trial 3 finished with value: 0.6172222222222222 and parameters: {'max_depth': 6, 'learning_rate': 0.01196284396631069, 'n_estimators': 635, 'subsample': 0.6614487471779947, 'colsample_bytree': 0.5812397569030672, 'min_child_weight': 6, 'gamma': 1.721685135171565, 'reg_alpha': 5.306265360372441e-07, 'reg_lambda': 0.004330073193969905, 'max_delta_step': 3}. Best is trial 3 with value: 0.6172222222222222.


Best trial: 3. Best value: 0.617222:   1%|          | 2/200 [00:08<13:09,  3.99s/it]

[I 2025-11-24 17:13:17,911] Trial 0 finished with value: 0.5 and parameters: {'max_depth': 4, 'learning_rate': 0.03923979307018735, 'n_estimators': 929, 'subsample': 0.809443122947326, 'colsample_bytree': 0.5466150783210233, 'min_child_weight': 8, 'gamma': 4.091851705279773, 'reg_alpha': 2.8119424382929515e-08, 'reg_lambda': 0.016565618232855024, 'max_delta_step': 3}. Best is trial 3 with value: 0.6172222222222222.


Best trial: 3. Best value: 0.617222:   2%|▏         | 3/200 [00:09<08:21,  2.54s/it]

...

Best trial: 6. Best value: 0.675278:   3%|▎         | 6/200 [00:17<09:20,  2.89s/it]

[I 2025-11-24 17:13:26,271] Trial 5 finished with value: 0.5801388888888889 and parameters: {'max_depth': 10, 'learning_rate': 0.013495223468829552, 'n_estimators': 918, 'subsample': 0.9818504401503849, 'colsample_bytree': 0.5037395916749248, 'min_child_weight': 7, 'gamma': 3.528735962575631, 'reg_alpha': 3.4616683042059764, 'reg_lambda': 0.06120979774644984, 'max_delta_step': 10}. Best is trial 6 with value: 0.6752777777777779.


Best trial: 6. Best value: 0.675278:   4%|▎         | 7/200 [00:19<09:07,  2.84s/it]

[I 2025-11-24 17:13:28,793] Trial 4 finished with value: 0.5441666666666667 and parameters: {'max_depth': 6, 'learning_rate': 0.02205229756199143, 'n_estimators': 1019, 'subsample': 0.6578735360708388, 'colsample_bytree': 0.581358333171203, 'min_child_weight': 1, 'gamma': 4.36838579630935, 'reg_alpha': 1.2161939018454574e-06, 'reg_lambda': 7.287010812607694e-05, 'max_delta_step': 4}. Best is trial 6 with value: 0.6752777777777779.


KeyboardInterrupt: 

In [822]:

datasets = task_hf_dfs_clean_lbd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_lbd_hc/xgb_results_handcrafted_extended",
    split_param="w.cz.fnusa",
)
print(summary)

[I 2025-11-24 17:13:45,302] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 1. Best value: 0.529167:   0%|          | 1/200 [00:04<15:54,  4.80s/it]

[I 2025-11-24 17:13:50,087] Trial 1 finished with value: 0.5291666666666666 and parameters: {'max_depth': 10, 'learning_rate': 0.1626070353011523, 'n_estimators': 511, 'subsample': 0.6893636086807002, 'colsample_bytree': 0.7981171149270851, 'min_child_weight': 3, 'gamma': 0.34672566743919886, 'reg_alpha': 0.00017826624906643214, 'reg_lambda': 0.00035088748020537316, 'max_delta_step': 3}. Best is trial 1 with value: 0.5291666666666666.


Best trial: 2. Best value: 0.536667:   1%|          | 2/200 [00:07<11:03,  3.35s/it]

[I 2025-11-24 17:13:52,428] Trial 2 finished with value: 0.5366666666666666 and parameters: {'max_depth': 4, 'learning_rate': 0.14727205416656491, 'n_estimators': 743, 'subsample': 0.9439178884377994, 'colsample_bytree': 0.5790785262081063, 'min_child_weight': 8, 'gamma': 3.2555108390948186, 'reg_alpha': 0.005570657691179944, 'reg_lambda': 1.1462157493344668e-05, 'max_delta_step': 3}. Best is trial 2 with value: 0.5366666666666666.


Best trial: 2. Best value: 0.536667:   2%|▏         | 3/200 [00:08<07:59,  2.43s/it]

...

Best trial: 2. Best value: 0.536667:   2%|▎         | 5/200 [00:12<07:09,  2.20s/it]

[I 2025-11-24 17:13:57,384] Trial 4 finished with value: 0.5275 and parameters: {'max_depth': 10, 'learning_rate': 0.16145305530550846, 'n_estimators': 1202, 'subsample': 0.8584465697631705, 'colsample_bytree': 0.5024248387609191, 'min_child_weight': 4, 'gamma': 3.8751283117115385, 'reg_alpha': 1.4539047634940673e-06, 'reg_lambda': 0.6704823234695475, 'max_delta_step': 1}. Best is trial 2 with value: 0.5366666666666666.


Best trial: 2. Best value: 0.536667:   3%|▎         | 6/200 [00:13<07:13,  2.23s/it]

[I 2025-11-24 17:13:58,692] Trial 5 finished with value: 0.5 and parameters: {'max_depth': 3, 'learning_rate': 0.012810444127514167, 'n_estimators': 1489, 'subsample': 0.8661905052459102, 'colsample_bytree': 0.8469075022771149, 'min_child_weight': 10, 'gamma': 3.7527688055852626, 'reg_alpha': 0.5806052163928499, 'reg_lambda': 0.2489213692987129, 'max_delta_step': 2}. Best is trial 2 with value: 0.5366666666666666.


KeyboardInterrupt: 

## MCI-AD vs HC -> OPTUNA + XGB

In [653]:

datasets = task_hf_compensated_comb_emb_lbl_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_handcrafted_combined_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)


[I 2025-11-24 16:40:18,685] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 3. Best value: 0.466667:   0%|          | 1/200 [00:06<22:53,  6.90s/it]

[I 2025-11-24 16:40:25,577] Trial 3 finished with value: 0.4666666666666667 and parameters: {'max_depth': 10, 'learning_rate': 0.10467233184386628, 'n_estimators': 476, 'subsample': 0.7548625428912235, 'colsample_bytree': 0.6343933536915265, 'min_child_weight': 1, 'gamma': 1.4291449257765898, 'reg_alpha': 2.263038788650688e-08, 'reg_lambda': 0.2847017979128992, 'max_delta_step': 5}. Best is trial 3 with value: 0.4666666666666667.


Best trial: 3. Best value: 0.466667:   1%|          | 2/200 [00:07<10:21,  3.14s/it]

[I 2025-11-24 16:40:26,078] Trial 2 finished with value: 0.4666666666666667 and parameters: {'max_depth': 5, 'learning_rate': 0.019650251461433645, 'n_estimators': 643, 'subsample': 0.9833964107581346, 'colsample_bytree': 0.5315848297508656, 'min_child_weight': 7, 'gamma': 2.1389374000181753, 'reg_alpha': 5.3792130325855816e-05, 'reg_lambda': 0.029864183713966416, 'max_delta_step': 9}. Best is trial 3 with value: 0.4666666666666667.


Best trial: 0. Best value: 0.479167:   2%|▏         | 3/200 [00:10<10:49,  3.30s/it]

...

Best trial: 4. Best value: 0.5:   4%|▎         | 7/200 [00:24<11:16,  3.50s/it]

[I 2025-11-24 16:40:43,305] Trial 6 finished with value: 0.4305555555555555 and parameters: {'max_depth': 10, 'learning_rate': 0.01720681224596904, 'n_estimators': 720, 'subsample': 0.7340995012643798, 'colsample_bytree': 0.7430148915200449, 'min_child_weight': 1, 'gamma': 0.054674244753114554, 'reg_alpha': 8.32620228556417e-06, 'reg_lambda': 1.0918150696730722e-05, 'max_delta_step': 4}. Best is trial 4 with value: 0.5.


Best trial: 4. Best value: 0.5:   4%|▍         | 8/200 [00:26<10:41,  3.34s/it]

[I 2025-11-24 16:40:45,423] Trial 7 finished with value: 0.4763888888888889 and parameters: {'max_depth': 3, 'learning_rate': 0.04571984141350797, 'n_estimators': 939, 'subsample': 0.7710937975709611, 'colsample_bytree': 0.9848939557875893, 'min_child_weight': 1, 'gamma': 0.8134062590517549, 'reg_alpha': 0.0035896265357103652, 'reg_lambda': 0.021634252100990778, 'max_delta_step': 10}. Best is trial 4 with value: 0.5.


KeyboardInterrupt: 

In [ ]:
task_hf_compensated_temp_emb_lbl_ad_hc.get('9_1')

In [654]:

datasets = task_hf_compensated_temp_emb_lbl_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_handcrafted_temporal_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)


[I 2025-11-24 16:40:57,343] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 0. Best value: 0.663889:   0%|          | 1/200 [00:08<29:29,  8.89s/it]

[I 2025-11-24 16:41:06,223] Trial 0 finished with value: 0.663888888888889 and parameters: {'max_depth': 4, 'learning_rate': 0.0698428126451406, 'n_estimators': 1039, 'subsample': 0.6570811578326257, 'colsample_bytree': 0.577759550339966, 'min_child_weight': 6, 'gamma': 3.562704984211908, 'reg_alpha': 3.5692000034466846, 'reg_lambda': 3.802350540579978, 'max_delta_step': 5}. Best is trial 0 with value: 0.663888888888889.


Best trial: 0. Best value: 0.663889:   1%|          | 2/200 [00:09<13:13,  4.01s/it]

[I 2025-11-24 16:41:06,816] Trial 1 finished with value: 0.5888888888888889 and parameters: {'max_depth': 3, 'learning_rate': 0.12963232903794134, 'n_estimators': 944, 'subsample': 0.7991235744833359, 'colsample_bytree': 0.577500315256529, 'min_child_weight': 1, 'gamma': 4.637473046688934, 'reg_alpha': 0.0012396793863840338, 'reg_lambda': 2.1978050689874347, 'max_delta_step': 10}. Best is trial 0 with value: 0.663888888888889.


Best trial: 0. Best value: 0.663889:   2%|▏         | 3/200 [00:11<09:37,  2.93s/it]

...

Best trial: 5. Best value: 0.688889:   4%|▎         | 7/200 [00:20<08:38,  2.69s/it]

[I 2025-11-24 16:41:18,053] Trial 6 finished with value: 0.575 and parameters: {'max_depth': 7, 'learning_rate': 0.016739917422999653, 'n_estimators': 632, 'subsample': 0.9089467374143059, 'colsample_bytree': 0.8006652338064985, 'min_child_weight': 1, 'gamma': 2.1842379838278756, 'reg_alpha': 7.369968696963068e-07, 'reg_lambda': 6.814729138280986e-05, 'max_delta_step': 2}. Best is trial 5 with value: 0.6888888888888889.


Best trial: 5. Best value: 0.688889:   4%|▍         | 8/200 [00:21<08:37,  2.69s/it]

[I 2025-11-24 16:41:18,878] Trial 7 finished with value: 0.5 and parameters: {'max_depth': 8, 'learning_rate': 0.261403468281551, 'n_estimators': 1134, 'subsample': 0.8633703505677333, 'colsample_bytree': 0.653418686165403, 'min_child_weight': 10, 'gamma': 3.5131013750818396, 'reg_alpha': 0.0001543814849514502, 'reg_lambda': 9.346941602297093e-08, 'max_delta_step': 2}. Best is trial 5 with value: 0.6888888888888889.


KeyboardInterrupt: 

In [655]:

datasets = task_hf_compensated_freq_emb_lbl_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_handcrafted_frerq_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)
## MCI-AD vs HC -> OPTUNA + XGB

[I 2025-11-24 16:41:23,934] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 3. Best value: 0.525:   0%|          | 1/200 [00:03<12:38,  3.81s/it]

[I 2025-11-24 16:41:27,736] Trial 3 finished with value: 0.525 and parameters: {'max_depth': 3, 'learning_rate': 0.10305185149218535, 'n_estimators': 262, 'subsample': 0.6110566924231683, 'colsample_bytree': 0.871184501155648, 'min_child_weight': 1, 'gamma': 2.9600164691564586, 'reg_alpha': 2.7343355878496006e-06, 'reg_lambda': 1.6877703959733927, 'max_delta_step': 3}. Best is trial 3 with value: 0.525.


Best trial: 3. Best value: 0.525:   1%|          | 2/200 [00:04<05:34,  1.69s/it]

[I 2025-11-24 16:41:27,937] Trial 1 finished with value: 0.5 and parameters: {'max_depth': 9, 'learning_rate': 0.09292170562593054, 'n_estimators': 394, 'subsample': 0.6584821911720878, 'colsample_bytree': 0.8247396794503881, 'min_child_weight': 8, 'gamma': 3.236170379849983, 'reg_alpha': 0.00015301832182753241, 'reg_lambda': 0.035538220406143246, 'max_delta_step': 8}. Best is trial 3 with value: 0.525.


Best trial: 3. Best value: 0.525:   2%|▏         | 3/200 [00:05<04:38,  1.41s/it]

...

Best trial: 3. Best value: 0.525:   3%|▎         | 6/200 [00:13<07:05,  2.19s/it]

[I 2025-11-24 16:41:37,384] Trial 4 finished with value: 0.5 and parameters: {'max_depth': 8, 'learning_rate': 0.08209710272613018, 'n_estimators': 996, 'subsample': 0.5012438979468736, 'colsample_bytree': 0.5386081730608407, 'min_child_weight': 7, 'gamma': 1.3639504289736544, 'reg_alpha': 1.2293482286598504e-07, 'reg_lambda': 7.293428663329567e-06, 'max_delta_step': 8}. Best is trial 3 with value: 0.525.


Best trial: 3. Best value: 0.525:   4%|▍         | 8/200 [00:16<06:34,  2.06s/it]


[I 2025-11-24 16:41:40,249] Trial 7 finished with value: 0.5083333333333333 and parameters: {'max_depth': 3, 'learning_rate': 0.024990196704342826, 'n_estimators': 601, 'subsample': 0.7351291667837958, 'colsample_bytree': 0.586121512318416, 'min_child_weight': 2, 'gamma': 1.6445036763711107, 'reg_alpha': 4.706481929304239e-05, 'reg_lambda': 0.9340817250054142, 'max_delta_step': 3}. Best is trial 3 with value: 0.525.
[I 2025-11-24 16:41:40,369] Trial 6 finished with value: 0.5 and parameters: {'max_depth': 6, 'learning_rate': 0.10566140693542732, 'n_estimators': 1211, 'subsample': 0.6881616337189094, 'colsample_bytree': 0.9833235324025702, 'min_child_weight': 2, 'gamma': 3.694674554743558, 'reg_alpha': 0.00024463493893949926, 'reg_lambda': 2.272934908022577e-08, 'max_delta_step': 1}. Best is trial 3 with value: 0.525.


KeyboardInterrupt: 

In [656]:
datasets = task_freq_dfs_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_freq_ad_hc_extended",
    split_param="emb_",
)
print(summary)


[I 2025-11-24 16:41:47,108] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 0. Best value: 0.5:   0%|          | 1/200 [00:03<12:39,  3.82s/it]

[I 2025-11-24 16:41:50,919] Trial 0 finished with value: 0.5 and parameters: {'max_depth': 8, 'learning_rate': 0.01561174896006025, 'n_estimators': 365, 'subsample': 0.6084421192384637, 'colsample_bytree': 0.736597814939281, 'min_child_weight': 8, 'gamma': 4.943722771140701, 'reg_alpha': 1.913491521024589e-05, 'reg_lambda': 0.6824844512353915, 'max_delta_step': 0}. Best is trial 0 with value: 0.5.


Best trial: 2. Best value: 0.55:   1%|          | 2/200 [00:10<17:35,  5.33s/it]

[I 2025-11-24 16:41:57,304] Trial 2 finished with value: 0.55 and parameters: {'max_depth': 8, 'learning_rate': 0.2019790302482448, 'n_estimators': 1097, 'subsample': 0.8430307726727864, 'colsample_bytree': 0.7699789744208072, 'min_child_weight': 4, 'gamma': 2.0684492955353107, 'reg_alpha': 0.000283961173528112, 'reg_lambda': 0.9865906645923982, 'max_delta_step': 4}. Best is trial 2 with value: 0.55.


Best trial: 1. Best value: 0.6125:   2%|▏         | 3/200 [00:10<10:00,  3.05s/it]

...

Best trial: 1. Best value: 0.6125:   4%|▍         | 9/200 [00:20<05:55,  1.86s/it]

[I 2025-11-24 16:42:07,853] Trial 5 finished with value: 0.4875 and parameters: {'max_depth': 6, 'learning_rate': 0.011329753177162026, 'n_estimators': 793, 'subsample': 0.6140575885191482, 'colsample_bytree': 0.8826954107324401, 'min_child_weight': 1, 'gamma': 4.097981688031693, 'reg_alpha': 0.06299373799067852, 'reg_lambda': 1.0985801496853088e-06, 'max_delta_step': 8}. Best is trial 1 with value: 0.6125.


Best trial: 1. Best value: 0.6125:   5%|▌         | 10/200 [00:21<06:45,  2.13s/it]

[I 2025-11-24 16:42:08,453] Trial 7 finished with value: 0.55 and parameters: {'max_depth': 7, 'learning_rate': 0.04924788998685576, 'n_estimators': 1353, 'subsample': 0.6703983051068823, 'colsample_bytree': 0.7554557981359256, 'min_child_weight': 5, 'gamma': 3.506139314530005, 'reg_alpha': 0.15157501706279977, 'reg_lambda': 1.8875266374224017e-05, 'max_delta_step': 5}. Best is trial 1 with value: 0.6125.


KeyboardInterrupt: 

In [657]:

# Temporal embedings
datasets = task_temp_dfs_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    #    scoring="balanced_accuracy",     # or "roc_auc"
    scoring="roc_auc",      
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_temporal_extended",
    split_param="emb_",
)
print(summary)


[I 2025-11-24 16:42:12,053] A new study created in memory with name: 1_1_roc_auc
Best trial: 3. Best value: 0.667361:   0%|          | 1/200 [00:04<13:49,  4.17s/it]

[I 2025-11-24 16:42:16,214] Trial 3 finished with value: 0.6673611111111111 and parameters: {'max_depth': 10, 'learning_rate': 0.020985206244008448, 'n_estimators': 352, 'subsample': 0.7276648683718488, 'colsample_bytree': 0.6324237276280075, 'min_child_weight': 3, 'gamma': 2.030715114686888, 'reg_alpha': 3.6805626133198455e-05, 'reg_lambda': 0.00145616843614587, 'max_delta_step': 3}. Best is trial 3 with value: 0.6673611111111111.


Best trial: 3. Best value: 0.667361:   1%|          | 2/200 [00:09<15:06,  4.58s/it]

[I 2025-11-24 16:42:21,075] Trial 2 finished with value: 0.5 and parameters: {'max_depth': 5, 'learning_rate': 0.08964454186405926, 'n_estimators': 914, 'subsample': 0.8105505441192127, 'colsample_bytree': 0.8591343271251082, 'min_child_weight': 10, 'gamma': 0.9630258740165737, 'reg_alpha': 0.99650085139425, 'reg_lambda': 0.0024824876214002856, 'max_delta_step': 0}. Best is trial 3 with value: 0.6673611111111111.


Best trial: 3. Best value: 0.667361:   2%|▏         | 3/200 [00:12<13:24,  4.09s/it]

...

Best trial: 4. Best value: 0.693519:   4%|▎         | 7/200 [00:19<07:03,  2.19s/it]

[I 2025-11-24 16:42:31,821] Trial 5 finished with value: 0.6212962962962962 and parameters: {'max_depth': 7, 'learning_rate': 0.16793234375743618, 'n_estimators': 1207, 'subsample': 0.6275580143718951, 'colsample_bytree': 0.7104212356608962, 'min_child_weight': 3, 'gamma': 2.331403392695628, 'reg_alpha': 0.0985125163049635, 'reg_lambda': 0.04928653064434507, 'max_delta_step': 0}. Best is trial 4 with value: 0.6935185185185185.
[I 2025-11-24 16:42:31,897] Trial 8 finished with value: 0.6685185185185185 and parameters: {'max_depth': 5, 'learning_rate': 0.014870022591234499, 'n_estimators': 354, 'subsample': 0.9819286501497126, 'colsample_bytree': 0.5971918973248115, 'min_child_weight': 7, 'gamma': 3.2634775202706083, 'reg_alpha': 0.01353276982396848, 'reg_lambda': 6.359530521389628e-07, 'max_delta_step': 3}. Best is trial 4 with value: 0.6935185185185185.


Best trial: 7. Best value: 0.694676:   4%|▍         | 9/200 [00:22<07:52,  2.47s/it]

[I 2025-11-24 16:42:34,302] Trial 7 finished with value: 0.694675925925926 and parameters: {'max_depth': 8, 'learning_rate': 0.02474595129551275, 'n_estimators': 1249, 'subsample': 0.9949970651436577, 'colsample_bytree': 0.8124125878888087, 'min_child_weight': 5, 'gamma': 1.7203225763361378, 'reg_alpha': 1.9141242843673853e-06, 'reg_lambda': 0.09739271465442297, 'max_delta_step': 0}. Best is trial 7 with value: 0.694675925925926.


KeyboardInterrupt: 

In [658]:

# Combined embedings
datasets = task_comb_dfs_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="roc_auc",     # or "roc_auc"
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_combined_extended",
    split_param="emb_",
)
print(summary)



[I 2025-11-24 16:42:40,196] A new study created in memory with name: 1_1_roc_auc
Best trial: 0. Best value: 0.5:   0%|          | 1/200 [00:03<12:46,  3.85s/it]

[I 2025-11-24 16:42:44,035] Trial 0 finished with value: 0.5 and parameters: {'max_depth': 8, 'learning_rate': 0.050640641814155306, 'n_estimators': 383, 'subsample': 0.6584400118736626, 'colsample_bytree': 0.589583985472514, 'min_child_weight': 10, 'gamma': 4.01970477543946, 'reg_alpha': 4.0485408404081574e-06, 'reg_lambda': 0.0028691612246080694, 'max_delta_step': 2}. Best is trial 0 with value: 0.5.


Best trial: 0. Best value: 0.5:   1%|          | 2/200 [00:04<05:43,  1.74s/it]

[I 2025-11-24 16:42:44,266] Trial 3 finished with value: 0.5 and parameters: {'max_depth': 6, 'learning_rate': 0.12481224254292711, 'n_estimators': 395, 'subsample': 0.6374742714718287, 'colsample_bytree': 0.6478326198814848, 'min_child_weight': 8, 'gamma': 4.127028576093012, 'reg_alpha': 1.775990472090842, 'reg_lambda': 4.513961570648278, 'max_delta_step': 8}. Best is trial 0 with value: 0.5.


Best trial: 0. Best value: 0.5:   2%|▏         | 3/200 [00:04<03:57,  1.21s/it]

...

Best trial: 1. Best value: 0.500231:   4%|▍         | 8/200 [00:17<07:40,  2.40s/it]

[I 2025-11-24 16:42:57,314] Trial 6 finished with value: 0.4104166666666667 and parameters: {'max_depth': 10, 'learning_rate': 0.0836537845047068, 'n_estimators': 1318, 'subsample': 0.6606226891267861, 'colsample_bytree': 0.6922608073345019, 'min_child_weight': 2, 'gamma': 2.683883006609789, 'reg_alpha': 0.34917267284652515, 'reg_lambda': 7.056359757087072e-05, 'max_delta_step': 10}. Best is trial 1 with value: 0.5002314814814814.


Best trial: 1. Best value: 0.500231:   4%|▍         | 9/200 [00:20<07:15,  2.28s/it]

[I 2025-11-24 16:43:00,733] Trial 7 finished with value: 0.4217592592592593 and parameters: {'max_depth': 7, 'learning_rate': 0.043338153558672696, 'n_estimators': 1194, 'subsample': 0.9907092005590494, 'colsample_bytree': 0.9432720829328851, 'min_child_weight': 4, 'gamma': 4.906898550830098, 'reg_alpha': 0.0003543005890539064, 'reg_lambda': 4.0211793289880505e-07, 'max_delta_step': 10}. Best is trial 1 with value: 0.5002314814814814.


KeyboardInterrupt: 

In [659]:

datasets = task_hf_dfs_clean_ad_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_handcrafted_extended",
    split_param="w.cz.fnusa",
)
print(summary)

[I 2025-11-24 16:43:07,813] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 2. Best value: 0.5:   0%|          | 1/200 [00:02<09:32,  2.88s/it]

[I 2025-11-24 16:43:10,672] Trial 2 finished with value: 0.5 and parameters: {'max_depth': 6, 'learning_rate': 0.019127601106354423, 'n_estimators': 219, 'subsample': 0.7657829493030086, 'colsample_bytree': 0.9149904068521211, 'min_child_weight': 10, 'gamma': 0.9905914199036242, 'reg_alpha': 6.573246932220573e-05, 'reg_lambda': 1.2090918576076068e-05, 'max_delta_step': 9}. Best is trial 2 with value: 0.5.


Best trial: 2. Best value: 0.5:   1%|          | 2/200 [00:06<11:08,  3.38s/it]

[I 2025-11-24 16:43:14,403] Trial 1 finished with value: 0.5 and parameters: {'max_depth': 3, 'learning_rate': 0.0582210478640884, 'n_estimators': 561, 'subsample': 0.5921998879134132, 'colsample_bytree': 0.9982122662637809, 'min_child_weight': 10, 'gamma': 0.893755469501531, 'reg_alpha': 1.0356102699429336e-07, 'reg_lambda': 2.804399125978178e-06, 'max_delta_step': 1}. Best is trial 2 with value: 0.5.


Best trial: 2. Best value: 0.5:   2%|▏         | 3/200 [00:06<06:24,  1.95s/it]

...

Best trial: 6. Best value: 0.5875:   4%|▎         | 7/200 [00:13<04:47,  1.49s/it]  

[I 2025-11-24 16:43:21,505] Trial 6 finished with value: 0.5875 and parameters: {'max_depth': 8, 'learning_rate': 0.03172059126325328, 'n_estimators': 731, 'subsample': 0.5225784639546407, 'colsample_bytree': 0.8799750905760324, 'min_child_weight': 1, 'gamma': 1.8124336578561921, 'reg_alpha': 1.4390688341851763e-08, 'reg_lambda': 2.2150222859758474e-08, 'max_delta_step': 8}. Best is trial 6 with value: 0.5875.


Best trial: 6. Best value: 0.5875:   4%|▍         | 8/200 [00:14<05:50,  1.82s/it]

[I 2025-11-24 16:43:22,397] Trial 5 finished with value: 0.5777777777777777 and parameters: {'max_depth': 5, 'learning_rate': 0.17308133913123563, 'n_estimators': 1309, 'subsample': 0.544298798631484, 'colsample_bytree': 0.9873732724698414, 'min_child_weight': 5, 'gamma': 1.8239629156186665, 'reg_alpha': 0.0017558421220043848, 'reg_lambda': 0.0014899083108730568, 'max_delta_step': 9}. Best is trial 6 with value: 0.5875.


KeyboardInterrupt: 

## MCI-LBD vs MCI-AD -> OTPUNA + XBG

In [661]:
task_hf_compensated_comb_emb_lbl_ad_lbd.get("1_1")

,label,subject,file_path,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,...,w.cz.fnusa.1_1_slope of duration of pen stops,w.cz.fnusa.1_1_slope of horizontal velocity (on-surface),w.cz.fnusa.1_1_slope of pressure,w.cz.fnusa.1_1_slope of velocity (on-surface),w.cz.fnusa.1_1_slope of vertical velocity (on-surface),w.cz.fnusa.1_1_spiral precision index,w.cz.fnusa.1_1_tightness of spiral,w.cz.fnusa.1_1_variability of spiral width,w.cz.fnusa.1_1_zero-crossing rate of spiral,diagnosis_y
0,1,COBEN-WTABLET-DM-AD15,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.070441,0.053146,0.024174,0.030326,0.001357,-0.032385,-0.038297,...,0.041338,0.012307,0.000107,0.017672,0.010502,18.809973,1.891182,0.090616,4.502542,0.0
1,1,COBEN-WTABLET-DV-AD04,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.071915,0.059692,0.024972,0.028608,0.000070,-0.028722,-0.041192,...,0.015836,0.006629,0.000076,0.008595,0.004086,16.379484,2.371073,0.099080,7.525870,0.0
2,1,COBEN-WTABLET-ER-AD09,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.066827,0.056126,0.029206,0.026343,0.006269,-0.035813,-0.042197,...,-0.007586,0.001641,0.000067,0.011007,0.009967,49.466945,2.456227,0.322380,13.943355,0.0
3,1,COBEN-WTABLET-ES-AD08,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.069645,0.053029,0.025364,0.030278,0.001204,-0.032683,-0.038478,...,0.026556,0.003282,0.000103,0.005476,0.003066,15.294063,2.380835,0.095670,7.709974,0.0
4,1,COBEN-WTABLET-ET-AD05,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.070062,0.051485,0.026057,0.031078,0.001382,-0.032270,-0.037237,...,0.067644,0.002180,-0.000008,0.004395,0.003140,10.166455,2.725233,0.082189,8.111747,0.0
5,1,COBEN-WTABLET-HS-AD06,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.069702,0.053135,0.026035,0.030166,0.001268,-0.032579,-0.038817,...,-0.007586,0.013491,0.000010,0.019669,0.011743,17.360028,2.000886,0.150766,4.997574,0.0
6,1,COBEN-WTABLET-KJ-AD13,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.070363,0.053997,0.025149,0.030960,0.000171,-0.031990,-0.038375,...,0.020669,0.015711,0.000151,0.025556,0.015500,15.930486,2.053615,0.085582,5.026178,0.0
7,1,COBEN-WTABLET-MM-AD07,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.071575,0.050625,0.024154,0.031817,-0.000752,-0.032361,-0.036412,...,0.012778,-0.000263,0.000143,0.001094,0.000827,22.826800,1.446711,0.075758,5.750165,0.0
8,1,COBEN-WTABLET-MS-AD10,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.071297,0.052086,0.024794,0.031838,-0.000934,-0.030865,-0.037217,...,-0.007586,0.017064,0.000177,0.028910,0.017972,20.659322,-1.484826,0.130360,5.741870,0.0
9,1,COBEN-WTABLET-OS-AD03,C:\dev\dolphin_initial_testing\DOLPHIN\out_pre...,0.070578,0.057131,0.023277,0.030778,-0.000636,-0.031124,-0.040026,...,-0.007586,0.009675,0.000095,0.013879,0.007869,21.272579,1.875281,0.183419,3.568202,0.0


In [823]:
#TODO: contains label
datasets = task_hf_compensated_comb_emb_lbl_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_combined_emb_compensated_ad_lbd_extended",
    split_param= "diagnosis_y",
)
print(summary)


[I 2025-11-24 17:14:27,640] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 1. Best value: 0.545:   0%|          | 1/200 [00:04<16:30,  4.98s/it]

[I 2025-11-24 17:14:32,609] Trial 1 finished with value: 0.545 and parameters: {'max_depth': 6, 'learning_rate': 0.022588416711810947, 'n_estimators': 384, 'subsample': 0.9253713434718522, 'colsample_bytree': 0.5715232704480742, 'min_child_weight': 1, 'gamma': 3.93739282712644, 'reg_alpha': 1.0419197597451308e-08, 'reg_lambda': 0.8823281986103492, 'max_delta_step': 0}. Best is trial 1 with value: 0.545.


Best trial: 1. Best value: 0.545:   1%|          | 2/200 [00:08<14:12,  4.30s/it]

[I 2025-11-24 17:14:36,443] Trial 2 finished with value: 0.5 and parameters: {'max_depth': 5, 'learning_rate': 0.025514770060788016, 'n_estimators': 820, 'subsample': 0.7487914052024434, 'colsample_bytree': 0.7677474370531434, 'min_child_weight': 10, 'gamma': 2.024363791223149, 'reg_alpha': 0.0001621453574195792, 'reg_lambda': 4.051725496258895e-08, 'max_delta_step': 5}. Best is trial 1 with value: 0.545.
[I 2025-11-24 17:14:36,526] Trial 3 finished with value: 0.5 and parameters: {'max_depth': 10, 'learning_rate': 0.29983011020940215, 'n_estimators': 812, 'subsample': 0.6797035107632288, 'colsample_bytree': 0.9940427936199383, 'min_child_weight': 5, 'gamma': 3.7944933497089735, 'reg_alpha': 1.1737556791908614e-06, 'reg_lambda': 8.796040438014843e-05, 'max_delta_step': 9}. Best is trial 1 with value: 0.545.


Best trial: 1. Best value: 0.545:   2%|▏         | 4/200 [00:10<07:28,  2.29s/it]

[I 2025-11-24 17:14:38,628] Trial 0 finished with value: 0.5 and parameters: {'max_depth': 9, 'learning_rate': 0.1050081175629831, 'n_estimators': 1328, 'subsample': 0.8720571389824596, 'colsample_bytree': 0.8070105152228415, 'min_child_weight': 8, 'gamma': 1.5205912049800752, 'reg_alpha': 8.439881106688614e-08, 'reg_lambda': 6.20093425749308e-06, 'max_delta_step': 9}. Best is trial 1 with value: 0.545.


Best trial: 1. Best value: 0.545:   2%|▎         | 5/200 [00:12<08:07,  2.50s/it]


[I 2025-11-24 17:14:40,127] Trial 4 finished with value: 0.5 and parameters: {'max_depth': 5, 'learning_rate': 0.05679849267691235, 'n_estimators': 1060, 'subsample': 0.6145390583322965, 'colsample_bytree': 0.9616406628820037, 'min_child_weight': 8, 'gamma': 4.882603459116977, 'reg_alpha': 1.0360695691193898e-05, 'reg_lambda': 1.714818830616261e-07, 'max_delta_step': 8}. Best is trial 1 with value: 0.545.


KeyboardInterrupt: 

In [824]:

datasets = task_hf_compensated_temp_emb_lbl_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_temporal_emb_compensated_ad_lbd_extended",
    split_param= "diagnosis_y",
)
print(summary)


[I 2025-11-24 17:14:45,089] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 2. Best value: 0.5:   0%|          | 1/200 [00:08<28:35,  8.62s/it]

[I 2025-11-24 17:14:53,701] Trial 2 finished with value: 0.5 and parameters: {'max_depth': 8, 'learning_rate': 0.04195078383149289, 'n_estimators': 955, 'subsample': 0.737910814072444, 'colsample_bytree': 0.8108994916004911, 'min_child_weight': 10, 'gamma': 3.0828184407780594, 'reg_alpha': 2.74138381395913e-06, 'reg_lambda': 0.001062477733162109, 'max_delta_step': 9}. Best is trial 2 with value: 0.5.


Best trial: 2. Best value: 0.5:   1%|          | 2/200 [00:09<13:54,  4.21s/it]

[I 2025-11-24 17:14:54,830] Trial 1 finished with value: 0.5 and parameters: {'max_depth': 7, 'learning_rate': 0.016633786159457986, 'n_estimators': 1089, 'subsample': 0.6896976101040122, 'colsample_bytree': 0.9695129078277491, 'min_child_weight': 7, 'gamma': 2.94941595284196, 'reg_alpha': 0.0002932414539751388, 'reg_lambda': 1.6379537140567432e-06, 'max_delta_step': 1}. Best is trial 2 with value: 0.5.


Best trial: 2. Best value: 0.5:   2%|▏         | 3/200 [00:10<08:20,  2.54s/it]

...

Best trial: 2. Best value: 0.5:  10%|█         | 20/200 [00:42<06:06,  2.04s/it]

[I 2025-11-24 17:15:27,365] Trial 19 finished with value: 0.5 and parameters: {'max_depth': 8, 'learning_rate': 0.03067030630555447, 'n_estimators': 1245, 'subsample': 0.653379109601648, 'colsample_bytree': 0.9298086436661467, 'min_child_weight': 5, 'gamma': 2.5173238237990563, 'reg_alpha': 1.8971189694161405e-05, 'reg_lambda': 5.1020050370165134e-05, 'max_delta_step': 9}. Best is trial 2 with value: 0.5.


Best trial: 2. Best value: 0.5:  10%|█         | 21/200 [00:42<06:06,  2.04s/it]

[I 2025-11-24 17:15:28,024] Trial 20 finished with value: 0.5 and parameters: {'max_depth': 8, 'learning_rate': 0.027731621529016575, 'n_estimators': 1188, 'subsample': 0.852429382245863, 'colsample_bytree': 0.9262167761102653, 'min_child_weight': 5, 'gamma': 2.685361695125572, 'reg_alpha': 2.5129707287085724e-05, 'reg_lambda': 7.122548122646088e-05, 'max_delta_step': 8}. Best is trial 2 with value: 0.5.


KeyboardInterrupt: 

In [825]:

datasets = task_hf_compensated_freq_emb_lbl_ad_lbl

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_frerq_emb_compensated_ad_lbd_extended",
    split_param= "diagnosis_y",
)
print(summary)
## MCI-AD vs HC -> OPTUNA + XGB

[I 2025-11-24 17:15:31,716] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 3. Best value: 0.5:   0%|          | 1/200 [00:04<16:05,  4.85s/it]

[I 2025-11-24 17:15:36,556] Trial 3 finished with value: 0.5 and parameters: {'max_depth': 8, 'learning_rate': 0.1650746780114246, 'n_estimators': 499, 'subsample': 0.9614979933172295, 'colsample_bytree': 0.5011228575205087, 'min_child_weight': 5, 'gamma': 1.287144873362453, 'reg_alpha': 1.3645791782353683e-06, 'reg_lambda': 0.00598774728536696, 'max_delta_step': 6}. Best is trial 3 with value: 0.5.


Best trial: 3. Best value: 0.5:   1%|          | 2/200 [00:10<16:39,  5.05s/it]

[I 2025-11-24 17:15:41,746] Trial 0 finished with value: 0.5 and parameters: {'max_depth': 8, 'learning_rate': 0.015517664072292443, 'n_estimators': 1111, 'subsample': 0.993638521056422, 'colsample_bytree': 0.6472772025738197, 'min_child_weight': 5, 'gamma': 0.0570912931545392, 'reg_alpha': 2.8253989621832374, 'reg_lambda': 3.371855805473888, 'max_delta_step': 0}. Best is trial 3 with value: 0.5.


Best trial: 3. Best value: 0.5:   2%|▏         | 3/200 [00:10<10:22,  3.16s/it]

...

Best trial: 8. Best value: 0.52:   6%|▌         | 11/200 [00:23<03:50,  1.22s/it]

[I 2025-11-24 17:15:55,340] Trial 11 finished with value: 0.5 and parameters: {'max_depth': 4, 'learning_rate': 0.26512603257131995, 'n_estimators': 405, 'subsample': 0.8762561104244221, 'colsample_bytree': 0.6576767075293933, 'min_child_weight': 10, 'gamma': 3.720197224072039, 'reg_alpha': 2.3977945272558268, 'reg_lambda': 2.269348074195758e-07, 'max_delta_step': 3}. Best is trial 8 with value: 0.5199999999999999.


Best trial: 8. Best value: 0.52:   6%|▌         | 12/200 [00:23<06:13,  1.99s/it]

[I 2025-11-24 17:15:55,532] Trial 10 finished with value: 0.5 and parameters: {'max_depth': 7, 'learning_rate': 0.017507380832490895, 'n_estimators': 456, 'subsample': 0.7734938265830651, 'colsample_bytree': 0.9118476262918045, 'min_child_weight': 5, 'gamma': 4.605539695864466, 'reg_alpha': 7.29456095361874e-07, 'reg_lambda': 0.14505942293339005, 'max_delta_step': 5}. Best is trial 8 with value: 0.5199999999999999.


KeyboardInterrupt: 

In [826]:
datasets = task_freq_dfs_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_lbd/xgb_results_freq_ad_lbd_extended",
    split_param="emb_",
)
print(summary)


[I 2025-11-24 17:16:00,844] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 2. Best value: 0.5:   0%|          | 1/200 [00:05<19:36,  5.91s/it]

[I 2025-11-24 17:16:06,747] Trial 2 finished with value: 0.5 and parameters: {'max_depth': 5, 'learning_rate': 0.04986956451120017, 'n_estimators': 667, 'subsample': 0.8314406107278586, 'colsample_bytree': 0.6630078783184415, 'min_child_weight': 6, 'gamma': 0.6264129709157151, 'reg_alpha': 0.024676246899165973, 'reg_lambda': 0.00033200394123849387, 'max_delta_step': 6}. Best is trial 2 with value: 0.5.


Best trial: 2. Best value: 0.5:   1%|          | 2/200 [00:06<08:37,  2.61s/it]

[I 2025-11-24 17:16:07,047] Trial 0 finished with value: 0.5 and parameters: {'max_depth': 8, 'learning_rate': 0.10202802022244815, 'n_estimators': 685, 'subsample': 0.5120435037273465, 'colsample_bytree': 0.661703889903573, 'min_child_weight': 8, 'gamma': 4.6066672949909115, 'reg_alpha': 9.938673008472714e-05, 'reg_lambda': 5.061970104705379, 'max_delta_step': 9}. Best is trial 2 with value: 0.5.


Best trial: 2. Best value: 0.5:   2%|▏         | 3/200 [00:07<06:18,  1.92s/it]

...

Best trial: 2. Best value: 0.5:   4%|▎         | 7/200 [00:17<05:59,  1.86s/it]

[I 2025-11-24 17:16:18,116] Trial 5 finished with value: 0.475 and parameters: {'max_depth': 9, 'learning_rate': 0.10171071042929228, 'n_estimators': 1237, 'subsample': 0.506549306252368, 'colsample_bytree': 0.5713506487727096, 'min_child_weight': 2, 'gamma': 4.144305575011742, 'reg_alpha': 1.1049762756020952e-06, 'reg_lambda': 1.2070833369191128e-05, 'max_delta_step': 9}. Best is trial 2 with value: 0.5.


Best trial: 2. Best value: 0.5:   4%|▍         | 8/200 [00:18<07:18,  2.29s/it]

[I 2025-11-24 17:16:19,129] Trial 6 finished with value: 0.46333333333333326 and parameters: {'max_depth': 10, 'learning_rate': 0.10366461368259623, 'n_estimators': 1481, 'subsample': 0.7274169288531019, 'colsample_bytree': 0.6574342259851241, 'min_child_weight': 2, 'gamma': 0.17983385437534638, 'reg_alpha': 0.0015605662318934127, 'reg_lambda': 2.7479808997428357e-06, 'max_delta_step': 0}. Best is trial 2 with value: 0.5.


KeyboardInterrupt: 

In [827]:

# Temporal embedings
datasets = task_temp_dfs_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    #    scoring="balanced_accuracy",     # or "roc_auc"
    scoring="roc_auc",      
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_lbd/xgb_results_temporal_ad_lbd_extended",
    split_param="emb_",
)
print(summary)


[I 2025-11-24 17:16:22,851] A new study created in memory with name: 1_1_roc_auc
Best trial: 3. Best value: 0.5:   0%|          | 1/200 [00:09<32:51,  9.91s/it]

[I 2025-11-24 17:16:32,747] Trial 3 finished with value: 0.5 and parameters: {'max_depth': 7, 'learning_rate': 0.05874144342575479, 'n_estimators': 1199, 'subsample': 0.5605871582621326, 'colsample_bytree': 0.5249726346564194, 'min_child_weight': 8, 'gamma': 3.901390312151813, 'reg_alpha': 0.003784664700048516, 'reg_lambda': 1.7100453581383957e-07, 'max_delta_step': 3}. Best is trial 3 with value: 0.5.
[I 2025-11-24 17:16:32,842] Trial 2 finished with value: 0.49111111111111116 and parameters: {'max_depth': 7, 'learning_rate': 0.014186512083435618, 'n_estimators': 979, 'subsample': 0.5935176907404499, 'colsample_bytree': 0.8746182548409085, 'min_child_weight': 1, 'gamma': 3.4534295164446016, 'reg_alpha': 0.14998459338308331, 'reg_lambda': 3.132091842387554e-08, 'max_delta_step': 10}. Best is trial 3 with value: 0.5.


Best trial: 3. Best value: 0.5:   2%|▏         | 3/200 [00:12<11:10,  3.40s/it]

[I 2025-11-24 17:16:34,999] Trial 5 finished with value: 0.5 and parameters: {'max_depth': 6, 'learning_rate': 0.0671407850518524, 'n_estimators': 219, 'subsample': 0.9288416435348502, 'colsample_bytree': 0.6653706421360303, 'min_child_weight': 6, 'gamma': 2.581340726637027, 'reg_alpha': 0.0007286115385477707, 'reg_lambda': 7.450127620387193e-08, 'max_delta_step': 2}. Best is trial 3 with value: 0.5.


Best trial: 3. Best value: 0.5:   2%|▏         | 4/200 [00:12<07:41,  2.36s/it]

...

Best trial: 3. Best value: 0.5:   4%|▍         | 8/200 [00:22<07:13,  2.26s/it]

[I 2025-11-24 17:16:44,993] Trial 8 finished with value: 0.5 and parameters: {'max_depth': 9, 'learning_rate': 0.09307119332962424, 'n_estimators': 1193, 'subsample': 0.6738279856192686, 'colsample_bytree': 0.5254579037478667, 'min_child_weight': 6, 'gamma': 2.929402021073988, 'reg_alpha': 0.04104596718051434, 'reg_lambda': 0.22267406989490263, 'max_delta_step': 10}. Best is trial 3 with value: 0.5.


Best trial: 3. Best value: 0.5:   4%|▍         | 9/200 [00:22<07:55,  2.49s/it]

[I 2025-11-24 17:16:45,266] Trial 7 finished with value: 0.5 and parameters: {'max_depth': 5, 'learning_rate': 0.07149044612176178, 'n_estimators': 1301, 'subsample': 0.770331834869411, 'colsample_bytree': 0.5613243054216159, 'min_child_weight': 5, 'gamma': 0.48360468043769644, 'reg_alpha': 1.0100327637360125e-08, 'reg_lambda': 1.649549471259225e-06, 'max_delta_step': 7}. Best is trial 3 with value: 0.5.


KeyboardInterrupt: 

In [828]:

# Combined embedings
datasets = task_comb_dfs_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="roc_auc",     # or "roc_auc"
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_lbd/xgb_results_combined_ad_lbd_extended",
    split_param="emb_",
)
print(summary)



[I 2025-11-24 17:16:51,299] A new study created in memory with name: 1_1_roc_auc
Best trial: 1. Best value: 0.5:   0%|          | 1/200 [00:03<12:45,  3.84s/it]

[I 2025-11-24 17:16:55,138] Trial 1 finished with value: 0.5 and parameters: {'max_depth': 10, 'learning_rate': 0.04546843940572257, 'n_estimators': 366, 'subsample': 0.5086766910700545, 'colsample_bytree': 0.9005270055590886, 'min_child_weight': 6, 'gamma': 4.088790890164831, 'reg_alpha': 2.4037373135804348e-08, 'reg_lambda': 3.666617734323458e-08, 'max_delta_step': 7}. Best is trial 1 with value: 0.5.


Best trial: 1. Best value: 0.5:   1%|          | 2/200 [00:07<12:26,  3.77s/it]

[I 2025-11-24 17:16:58,857] Trial 0 finished with value: 0.5 and parameters: {'max_depth': 4, 'learning_rate': 0.054064843229700794, 'n_estimators': 741, 'subsample': 0.9568460222980795, 'colsample_bytree': 0.8593142341708306, 'min_child_weight': 9, 'gamma': 3.24523762861286, 'reg_alpha': 9.393830306710115, 'reg_lambda': 1.3583887289184016, 'max_delta_step': 0}. Best is trial 1 with value: 0.5.


Best trial: 1. Best value: 0.5:   2%|▏         | 3/200 [00:08<07:32,  2.30s/it]

...

Best trial: 3. Best value: 0.627778:   4%|▍         | 8/200 [00:14<03:40,  1.15s/it]

[I 2025-11-24 17:17:05,272] Trial 8 finished with value: 0.5 and parameters: {'max_depth': 7, 'learning_rate': 0.04935104132054618, 'n_estimators': 358, 'subsample': 0.9837973788171797, 'colsample_bytree': 0.8196801592606631, 'min_child_weight': 10, 'gamma': 2.2285236174816294, 'reg_alpha': 4.973517840093235e-05, 'reg_lambda': 4.671026839522279e-08, 'max_delta_step': 6}. Best is trial 3 with value: 0.6277777777777778.
[I 2025-11-24 17:17:05,403] Trial 5 finished with value: 0.4966666666666667 and parameters: {'max_depth': 5, 'learning_rate': 0.010097882894400285, 'n_estimators': 583, 'subsample': 0.6057637522691002, 'colsample_bytree': 0.8332993268424316, 'min_child_weight': 3, 'gamma': 2.015834474924165, 'reg_alpha': 1.8738704962174018, 'reg_lambda': 0.00023142682937022648, 'max_delta_step': 8}. Best is trial 3 with value: 0.6277777777777778.


Best trial: 3. Best value: 0.627778:   4%|▍         | 9/200 [00:15<05:34,  1.75s/it]


[I 2025-11-24 17:17:07,067] Trial 6 finished with value: 0.5 and parameters: {'max_depth': 10, 'learning_rate': 0.02443670281306392, 'n_estimators': 1051, 'subsample': 0.9021285276497737, 'colsample_bytree': 0.7938511536351208, 'min_child_weight': 9, 'gamma': 1.3563368299647687, 'reg_alpha': 9.394332793580073e-08, 'reg_lambda': 5.291725399267209e-06, 'max_delta_step': 6}. Best is trial 3 with value: 0.6277777777777778.


KeyboardInterrupt: 

In [829]:

datasets = task_hf_dfs_clean_ad_lbd

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_lbd/xgb_results_handcrafted_ad_lbd_extended",
    split_param="w.cz.fnusa",
)
print(summary)

[I 2025-11-24 17:17:10,883] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 3. Best value: 0.5:   0%|          | 1/200 [00:05<17:32,  5.29s/it]

[I 2025-11-24 17:17:16,165] Trial 3 finished with value: 0.5 and parameters: {'max_depth': 8, 'learning_rate': 0.0715338762356726, 'n_estimators': 599, 'subsample': 0.6804248446428706, 'colsample_bytree': 0.6981662542613263, 'min_child_weight': 5, 'gamma': 3.2203480267792277, 'reg_alpha': 5.887133862720844e-05, 'reg_lambda': 0.0015010021580102821, 'max_delta_step': 10}. Best is trial 3 with value: 0.5.


Best trial: 0. Best value: 0.626667:   1%|          | 2/200 [00:07<10:52,  3.30s/it]

[I 2025-11-24 17:17:18,065] Trial 0 finished with value: 0.6266666666666666 and parameters: {'max_depth': 6, 'learning_rate': 0.012208369411603307, 'n_estimators': 748, 'subsample': 0.8271834324165506, 'colsample_bytree': 0.5385360735970438, 'min_child_weight': 1, 'gamma': 0.37169440713006596, 'reg_alpha': 0.0009491421429461293, 'reg_lambda': 1.0006318832042245, 'max_delta_step': 3}. Best is trial 0 with value: 0.6266666666666666.
[I 2025-11-24 17:17:18,125] Trial 2 finished with value: 0.5733333333333334 and parameters: {'max_depth': 7, 'learning_rate': 0.043225063812290215, 'n_estimators': 794, 'subsample': 0.6494768337626744, 'colsample_bytree': 0.7999869576653316, 'min_child_weight': 2, 'gamma': 2.542334075305411, 'reg_alpha': 0.41451078443732287, 'reg_lambda': 7.10590576926001, 'max_delta_step': 1}. Best is trial 0 with value: 0.6266666666666666.


Best trial: 0. Best value: 0.626667:   2%|▏         | 4/200 [00:09<06:15,  1.92s/it]

...

Best trial: 0. Best value: 0.626667:   4%|▍         | 8/200 [00:18<06:28,  2.02s/it]

[I 2025-11-24 17:17:29,290] Trial 8 finished with value: 0.5 and parameters: {'max_depth': 9, 'learning_rate': 0.024980663798998447, 'n_estimators': 1030, 'subsample': 0.8738301009959817, 'colsample_bytree': 0.7610164455773609, 'min_child_weight': 7, 'gamma': 3.9357076028203797, 'reg_alpha': 0.0001521035303090236, 'reg_lambda': 3.731287106016917e-05, 'max_delta_step': 8}. Best is trial 0 with value: 0.6266666666666666.


Best trial: 0. Best value: 0.626667:   4%|▍         | 9/200 [00:18<06:36,  2.07s/it]

[I 2025-11-24 17:17:29,547] Trial 7 finished with value: 0.555 and parameters: {'max_depth': 3, 'learning_rate': 0.018252545295259798, 'n_estimators': 1279, 'subsample': 0.8464871475875707, 'colsample_bytree': 0.649397674541798, 'min_child_weight': 3, 'gamma': 1.2070274382061892, 'reg_alpha': 3.385738382682348e-05, 'reg_lambda': 1.011874458801992e-08, 'max_delta_step': 3}. Best is trial 0 with value: 0.6266666666666666.


KeyboardInterrupt: 

## MCI-PD vs HC -> OPTUNA + XGB

In [830]:
#TODO: [0,4] in diagnosis
datasets = task_hf_compensated_comb_emb_lbl_pd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_pd_hc/xgb_results_handcrafted_combined_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)


[I 2025-11-24 17:17:33,427] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 0. Best value: 0.5:   0%|          | 1/200 [00:05<17:14,  5.20s/it]

[I 2025-11-24 17:17:38,613] Trial 0 finished with value: 0.5 and parameters: {'max_depth': 8, 'learning_rate': 0.02020287261092889, 'n_estimators': 536, 'subsample': 0.7122841585402435, 'colsample_bytree': 0.6572367136960254, 'min_child_weight': 10, 'gamma': 1.1054211759575732, 'reg_alpha': 0.689097168475902, 'reg_lambda': 1.5806348644482056, 'max_delta_step': 1}. Best is trial 0 with value: 0.5.


Best trial: 1. Best value: 0.513889:   1%|          | 2/200 [00:06<09:54,  3.00s/it]

[I 2025-11-24 17:17:40,081] Trial 1 finished with value: 0.513888888888889 and parameters: {'max_depth': 8, 'learning_rate': 0.019531225884482164, 'n_estimators': 545, 'subsample': 0.8775644100280602, 'colsample_bytree': 0.5432523794181509, 'min_child_weight': 4, 'gamma': 2.873884954607849, 'reg_alpha': 0.0017740945385606118, 'reg_lambda': 1.213901761309368e-07, 'max_delta_step': 8}. Best is trial 1 with value: 0.513888888888889.


Best trial: 1. Best value: 0.513889:   2%|▏         | 3/200 [00:10<11:50,  3.60s/it]

...

Best trial: 1. Best value: 0.513889:   4%|▍         | 8/200 [00:17<04:41,  1.47s/it]

[I 2025-11-24 17:17:51,165] Trial 6 finished with value: 0.5013888888888889 and parameters: {'max_depth': 9, 'learning_rate': 0.015640508152350584, 'n_estimators': 330, 'subsample': 0.8351147268883254, 'colsample_bytree': 0.9894672266060974, 'min_child_weight': 3, 'gamma': 2.3801187338272327, 'reg_alpha': 0.30921071884421375, 'reg_lambda': 1.4655197115434942e-08, 'max_delta_step': 3}. Best is trial 1 with value: 0.513888888888889.


Best trial: 7. Best value: 0.526389:   4%|▍         | 9/200 [00:23<08:21,  2.62s/it]

[I 2025-11-24 17:17:57,044] Trial 7 finished with value: 0.5263888888888889 and parameters: {'max_depth': 7, 'learning_rate': 0.13838331161595463, 'n_estimators': 1455, 'subsample': 0.8410181568172654, 'colsample_bytree': 0.5794798186614549, 'min_child_weight': 2, 'gamma': 3.9845370786112295, 'reg_alpha': 0.04397365200093401, 'reg_lambda': 1.449286777634413e-05, 'max_delta_step': 0}. Best is trial 7 with value: 0.5263888888888889.


KeyboardInterrupt: 

In [831]:

datasets = task_hf_compensated_temp_emb_lbl_pd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_pd_hc/xgb_results_handcrafted_temporal_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)


[I 2025-11-24 17:18:08,515] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 2. Best value: 0.4875:   0%|          | 1/200 [00:04<13:36,  4.11s/it]

[I 2025-11-24 17:18:12,610] Trial 2 finished with value: 0.4875 and parameters: {'max_depth': 5, 'learning_rate': 0.13663997360130384, 'n_estimators': 356, 'subsample': 0.6926213947689557, 'colsample_bytree': 0.5013272919800564, 'min_child_weight': 5, 'gamma': 0.37604571871399595, 'reg_alpha': 1.577195846458131e-05, 'reg_lambda': 0.06305671050849494, 'max_delta_step': 3}. Best is trial 2 with value: 0.4875.


Best trial: 1. Best value: 0.5:   1%|          | 2/200 [00:05<08:13,  2.49s/it]   

[I 2025-11-24 17:18:13,975] Trial 1 finished with value: 0.5 and parameters: {'max_depth': 6, 'learning_rate': 0.013251698330920065, 'n_estimators': 595, 'subsample': 0.63704649476844, 'colsample_bytree': 0.7374237240499708, 'min_child_weight': 8, 'gamma': 4.115595875324885, 'reg_alpha': 3.2134854199567844e-05, 'reg_lambda': 5.516477381448431, 'max_delta_step': 9}. Best is trial 1 with value: 0.5.


Best trial: 0. Best value: 0.551389:   2%|▏         | 3/200 [00:07<08:01,  2.45s/it]

[I 2025-11-24 17:18:16,362] Trial 0 finished with value: 0.5513888888888889 and parameters: {'max_depth': 3, 'learning_rate': 0.015344195782117829, 'n_estimators': 867, 'subsample': 0.7265289891914788, 'colsample_bytree': 0.6055886802511556, 'min_child_weight': 3, 'gamma': 4.824661727846781, 'reg_alpha': 0.001583527995542847, 'reg_lambda': 6.010196609227678e-07, 'max_delta_step': 3}. Best is trial 0 with value: 0.5513888888888889.


Best trial: 0. Best value: 0.551389:   2%|▏         | 4/200 [00:08<06:52,  2.10s/it]

[I 2025-11-24 17:18:16,911] Trial 3 finished with value: 0.5375 and parameters: {'max_depth': 7, 'learning_rate': 0.04113246576522367, 'n_estimators': 1150, 'subsample': 0.8263215428950581, 'colsample_bytree': 0.8717884856185139, 'min_child_weight': 6, 'gamma': 3.513243280552084, 'reg_alpha': 0.3193010079810307, 'reg_lambda': 1.3583016734863259e-06, 'max_delta_step': 1}. Best is trial 0 with value: 0.5513888888888889.


KeyboardInterrupt: 

In [832]:

datasets = task_hf_compensated_freq_emb_lbl_pd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_pd_hc/xgb_results_handcrafted_frerq_emb_compensated_extended",
    split_param= "diagnosis_y",
)
print(summary)
## MCI-AD vs HC -> OPTUNA + XGB

[I 2025-11-24 17:18:27,793] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 0. Best value: 0.5:   0%|          | 1/200 [00:06<22:06,  6.67s/it]

[I 2025-11-24 17:18:34,445] Trial 0 finished with value: 0.5 and parameters: {'max_depth': 10, 'learning_rate': 0.13871123453167786, 'n_estimators': 669, 'subsample': 0.8792502916106264, 'colsample_bytree': 0.9460090235909211, 'min_child_weight': 9, 'gamma': 0.9830357337411827, 'reg_alpha': 3.930921990972083e-06, 'reg_lambda': 1.8914809361928822, 'max_delta_step': 9}. Best is trial 0 with value: 0.5.


Best trial: 0. Best value: 0.5:   1%|          | 2/200 [00:10<16:27,  4.99s/it]

[I 2025-11-24 17:18:38,256] Trial 3 finished with value: 0.44027777777777777 and parameters: {'max_depth': 8, 'learning_rate': 0.2690504185845462, 'n_estimators': 1078, 'subsample': 0.7731944718160677, 'colsample_bytree': 0.6401895542094811, 'min_child_weight': 6, 'gamma': 2.8998262312586216, 'reg_alpha': 0.0005581326967752799, 'reg_lambda': 2.0419093560629938e-08, 'max_delta_step': 3}. Best is trial 0 with value: 0.5.


Best trial: 1. Best value: 0.533333:   2%|▏         | 3/200 [00:11<10:54,  3.32s/it]

...

Best trial: 4. Best value: 0.534722:   2%|▏         | 4/200 [00:13<09:04,  2.78s/it]

[I 2025-11-24 17:18:41,551] Trial 4 finished with value: 0.5347222222222222 and parameters: {'max_depth': 10, 'learning_rate': 0.14276011307433717, 'n_estimators': 866, 'subsample': 0.9884346099232536, 'colsample_bytree': 0.7251619976013866, 'min_child_weight': 5, 'gamma': 2.1173030015863548, 'reg_alpha': 0.5619924785216883, 'reg_lambda': 0.00037111763932256366, 'max_delta_step': 9}. Best is trial 4 with value: 0.5347222222222222.


Best trial: 2. Best value: 0.545833:   2%|▎         | 5/200 [00:15<10:08,  3.12s/it]


[I 2025-11-24 17:18:43,389] Trial 2 finished with value: 0.5458333333333333 and parameters: {'max_depth': 8, 'learning_rate': 0.10877350914734349, 'n_estimators': 1425, 'subsample': 0.7772742602282436, 'colsample_bytree': 0.8682487546273074, 'min_child_weight': 1, 'gamma': 2.6306620883838523, 'reg_alpha': 1.5390989158227952e-05, 'reg_lambda': 5.916014368644124, 'max_delta_step': 9}. Best is trial 2 with value: 0.5458333333333333.


KeyboardInterrupt: 

In [833]:
datasets = task_freq_dfs_pd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_pd_hc/xgb_results_freq_pd_hc_extended",
    split_param="emb_",
)
print(summary)


[I 2025-11-24 17:18:46,862] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 1. Best value: 0.484722:   0%|          | 1/200 [00:07<24:54,  7.51s/it]

[I 2025-11-24 17:18:54,362] Trial 1 finished with value: 0.4847222222222222 and parameters: {'max_depth': 8, 'learning_rate': 0.2516824209280866, 'n_estimators': 653, 'subsample': 0.8342621787262646, 'colsample_bytree': 0.5883072748458267, 'min_child_weight': 2, 'gamma': 4.7113159604527155, 'reg_alpha': 1.632811214780103e-07, 'reg_lambda': 0.20496521321902778, 'max_delta_step': 3}. Best is trial 1 with value: 0.4847222222222222.


Best trial: 3. Best value: 0.5:   1%|          | 2/200 [00:10<16:46,  5.08s/it]     

[I 2025-11-24 17:18:57,744] Trial 3 finished with value: 0.5 and parameters: {'max_depth': 5, 'learning_rate': 0.010062580517602697, 'n_estimators': 1141, 'subsample': 0.9182231614851033, 'colsample_bytree': 0.6825240872005001, 'min_child_weight': 10, 'gamma': 2.8657968418379753, 'reg_alpha': 2.1937143334792816e-08, 'reg_lambda': 7.24487255604052e-07, 'max_delta_step': 6}. Best is trial 3 with value: 0.5.


Best trial: 3. Best value: 0.5:   2%|▏         | 4/200 [00:11<06:18,  1.93s/it]

[I 2025-11-24 17:18:58,450] Trial 2 finished with value: 0.5 and parameters: {'max_depth': 9, 'learning_rate': 0.16989756405534875, 'n_estimators': 1188, 'subsample': 0.9456286422236821, 'colsample_bytree': 0.6207843684419213, 'min_child_weight': 9, 'gamma': 2.2585520819771405, 'reg_alpha': 0.00026224415084487236, 'reg_lambda': 4.026170636643082e-06, 'max_delta_step': 3}. Best is trial 3 with value: 0.5.
[I 2025-11-24 17:18:58,615] Trial 0 finished with value: 0.4375 and parameters: {'max_depth': 4, 'learning_rate': 0.04140723418250094, 'n_estimators': 1211, 'subsample': 0.5254631515743635, 'colsample_bytree': 0.7612716794986742, 'min_child_weight': 6, 'gamma': 4.603376756532233, 'reg_alpha': 2.5066184330926813, 'reg_lambda': 0.03061368863716452, 'max_delta_step': 1}. Best is trial 3 with value: 0.5.


Best trial: 3. Best value: 0.5:   2%|▎         | 5/200 [00:14<09:21,  2.88s/it]

[I 2025-11-24 17:19:01,243] Trial 4 finished with value: 0.4097222222222222 and parameters: {'max_depth': 6, 'learning_rate': 0.1092695494757908, 'n_estimators': 871, 'subsample': 0.9870103396678345, 'colsample_bytree': 0.7852610817193817, 'min_child_weight': 2, 'gamma': 2.1188648668435057, 'reg_alpha': 5.316156980915433, 'reg_lambda': 0.00033303755473692594, 'max_delta_step': 1}. Best is trial 3 with value: 0.5.


KeyboardInterrupt: 

In [834]:

# Temporal embedings
datasets = task_temp_dfs_pd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    #    scoring="balanced_accuracy",     # or "roc_auc"
    scoring="roc_auc",      
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_temporal_extended",
    split_param="emb_",
)
print(summary)


[I 2025-11-24 17:19:05,173] A new study created in memory with name: 1_1_roc_auc
Best trial: 0. Best value: 0.5:   0%|          | 1/200 [00:04<15:05,  4.55s/it]

[I 2025-11-24 17:19:09,707] Trial 0 finished with value: 0.5 and parameters: {'max_depth': 4, 'learning_rate': 0.11843565988844032, 'n_estimators': 477, 'subsample': 0.7639719207367217, 'colsample_bytree': 0.8967870672758734, 'min_child_weight': 10, 'gamma': 1.9385938270633534, 'reg_alpha': 0.0005765091122354244, 'reg_lambda': 0.6655237608382015, 'max_delta_step': 1}. Best is trial 0 with value: 0.5.


Best trial: 2. Best value: 0.518519:   1%|          | 2/200 [00:05<08:24,  2.55s/it]

[I 2025-11-24 17:19:10,861] Trial 2 finished with value: 0.5185185185185185 and parameters: {'max_depth': 4, 'learning_rate': 0.237180883886622, 'n_estimators': 558, 'subsample': 0.8314281005507586, 'colsample_bytree': 0.84809529610519, 'min_child_weight': 3, 'gamma': 2.2549177520265005, 'reg_alpha': 0.018217789606743622, 'reg_lambda': 1.1270765843615136, 'max_delta_step': 7}. Best is trial 2 with value: 0.5185185185185185.


Best trial: 1. Best value: 0.584491:   2%|▏         | 3/200 [00:06<05:18,  1.62s/it]

...

Best trial: 1. Best value: 0.584491:   3%|▎         | 6/200 [00:16<07:18,  2.26s/it]

[I 2025-11-24 17:19:21,246] Trial 4 finished with value: 0.5 and parameters: {'max_depth': 4, 'learning_rate': 0.022016981315600738, 'n_estimators': 1287, 'subsample': 0.8071068461214703, 'colsample_bytree': 0.7487394711221045, 'min_child_weight': 8, 'gamma': 3.9952856952469746, 'reg_alpha': 1.882539226668549e-05, 'reg_lambda': 0.09652660907885051, 'max_delta_step': 5}. Best is trial 1 with value: 0.5844907407407407.


Best trial: 1. Best value: 0.584491:   4%|▎         | 7/200 [00:17<08:00,  2.49s/it]

[I 2025-11-24 17:19:22,591] Trial 5 finished with value: 0.5 and parameters: {'max_depth': 7, 'learning_rate': 0.12381182676721436, 'n_estimators': 1459, 'subsample': 0.8819857721424156, 'colsample_bytree': 0.833049304110719, 'min_child_weight': 9, 'gamma': 3.0578361793485627, 'reg_alpha': 8.722096377183232e-07, 'reg_lambda': 0.023942188945373512, 'max_delta_step': 8}. Best is trial 1 with value: 0.5844907407407407.


KeyboardInterrupt: 

In [835]:

# Combined embedings
datasets = task_comb_dfs_pd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="roc_auc",     # or "roc_auc"
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_combined_extended",
    split_param="emb_",
)
print(summary)



[I 2025-11-24 17:19:32,334] A new study created in memory with name: 1_1_roc_auc
Best trial: 2. Best value: 0.533796:   0%|          | 1/200 [00:09<30:15,  9.13s/it]

[I 2025-11-24 17:19:41,451] Trial 2 finished with value: 0.5337962962962963 and parameters: {'max_depth': 5, 'learning_rate': 0.04428016252224163, 'n_estimators': 717, 'subsample': 0.9991599557697408, 'colsample_bytree': 0.8984039533812491, 'min_child_weight': 5, 'gamma': 3.830655735538417, 'reg_alpha': 6.511054520692613e-06, 'reg_lambda': 9.715511301640574e-06, 'max_delta_step': 7}. Best is trial 2 with value: 0.5337962962962963.


Best trial: 2. Best value: 0.533796:   1%|          | 2/200 [00:10<14:27,  4.38s/it]

[I 2025-11-24 17:19:42,511] Trial 3 finished with value: 0.5 and parameters: {'max_depth': 9, 'learning_rate': 0.09883800816949503, 'n_estimators': 1066, 'subsample': 0.751707686181011, 'colsample_bytree': 0.5450102437454555, 'min_child_weight': 10, 'gamma': 0.19344670960686927, 'reg_alpha': 9.005655870153504, 'reg_lambda': 0.00025708452410749566, 'max_delta_step': 1}. Best is trial 2 with value: 0.5337962962962963.


Best trial: 2. Best value: 0.533796:   2%|▏         | 3/200 [00:10<08:45,  2.67s/it]

...

Best trial: 1. Best value: 0.547917:   2%|▎         | 5/200 [00:21<14:43,  4.53s/it]

[I 2025-11-24 17:19:53,416] Trial 6 finished with value: 0.5365740740740741 and parameters: {'max_depth': 5, 'learning_rate': 0.0659490927218734, 'n_estimators': 1184, 'subsample': 0.6938967554327251, 'colsample_bytree': 0.8967036915684674, 'min_child_weight': 6, 'gamma': 1.0600575009612663, 'reg_alpha': 1.6951354201716647e-05, 'reg_lambda': 0.00245563210819265, 'max_delta_step': 10}. Best is trial 1 with value: 0.5479166666666667.


Best trial: 1. Best value: 0.547917:   4%|▎         | 7/200 [00:22<10:14,  3.18s/it]


[I 2025-11-24 17:19:54,451] Trial 4 finished with value: 0.5 and parameters: {'max_depth': 9, 'learning_rate': 0.011591992885355428, 'n_estimators': 1468, 'subsample': 0.6586998853669648, 'colsample_bytree': 0.8724302726691149, 'min_child_weight': 9, 'gamma': 0.7579807348111395, 'reg_alpha': 0.01638253021374122, 'reg_lambda': 0.0002844047724657039, 'max_delta_step': 3}. Best is trial 1 with value: 0.5479166666666667.
[I 2025-11-24 17:19:54,588] Trial 5 finished with value: 0.5412037037037037 and parameters: {'max_depth': 4, 'learning_rate': 0.12806766822505442, 'n_estimators': 920, 'subsample': 0.594335826010232, 'colsample_bytree': 0.7620470929149826, 'min_child_weight': 1, 'gamma': 0.030943872881149526, 'reg_alpha': 0.41669815491958806, 'reg_lambda': 4.1283345708212815, 'max_delta_step': 6}. Best is trial 1 with value: 0.5479166666666667.


KeyboardInterrupt: 

In [836]:

datasets = task_hf_dfs_clean_pd_hc

summary, models, studies = optimize_many(
    datasets,
    x_y_split_fn=x_y_split,          # your splitter
    scoring="balanced_accuracy",     # or "roc_auc"
    #scoring="roc_auc",    
    n_trials=200,
    n_splits=5,
    output_dir="xgb_ad_hc/xgb_results_handcrafted_extended",
    split_param="w.cz.fnusa",
)
print(summary)

[I 2025-11-24 17:19:59,022] A new study created in memory with name: 1_1_balanced_accuracy
Best trial: 3. Best value: 0.455556:   0%|          | 1/200 [00:03<13:09,  3.97s/it]

[I 2025-11-24 17:20:02,981] Trial 3 finished with value: 0.45555555555555555 and parameters: {'max_depth': 6, 'learning_rate': 0.03150171914955825, 'n_estimators': 346, 'subsample': 0.5989155034839365, 'colsample_bytree': 0.9464539795697532, 'min_child_weight': 1, 'gamma': 1.724983460818481, 'reg_alpha': 5.196650417423012e-08, 'reg_lambda': 2.03392009652842e-06, 'max_delta_step': 3}. Best is trial 3 with value: 0.45555555555555555.


Best trial: 1. Best value: 0.511111:   1%|          | 2/200 [00:09<16:51,  5.11s/it]

[I 2025-11-24 17:20:08,885] Trial 1 finished with value: 0.5111111111111111 and parameters: {'max_depth': 8, 'learning_rate': 0.2080287551873214, 'n_estimators': 1100, 'subsample': 0.6554739849387072, 'colsample_bytree': 0.9637098186120155, 'min_child_weight': 1, 'gamma': 3.086273281368213, 'reg_alpha': 0.00038865444766801225, 'reg_lambda': 0.0038348169282090812, 'max_delta_step': 6}. Best is trial 1 with value: 0.5111111111111111.


Best trial: 1. Best value: 0.511111:   2%|▏         | 3/200 [00:10<10:44,  3.27s/it]

...

Best trial: 4. Best value: 0.551389:   3%|▎         | 6/200 [00:14<05:33,  1.72s/it]

[I 2025-11-24 17:20:13,771] Trial 5 finished with value: 0.5 and parameters: {'max_depth': 5, 'learning_rate': 0.15235730465659425, 'n_estimators': 659, 'subsample': 0.572430978364874, 'colsample_bytree': 0.7179928594271845, 'min_child_weight': 7, 'gamma': 3.5728173435421073, 'reg_alpha': 5.562895776168737e-06, 'reg_lambda': 0.0038823458365152687, 'max_delta_step': 1}. Best is trial 1 with value: 0.5111111111111111.
[I 2025-11-24 17:20:13,943] Trial 4 finished with value: 0.5513888888888889 and parameters: {'max_depth': 8, 'learning_rate': 0.2241314813661879, 'n_estimators': 1262, 'subsample': 0.894832887448767, 'colsample_bytree': 0.8469730664521631, 'min_child_weight': 5, 'gamma': 1.7459152402246292, 'reg_alpha': 0.014439524777353444, 'reg_lambda': 0.00030210794881652505, 'max_delta_step': 7}. Best is trial 4 with value: 0.5513888888888889.


Best trial: 4. Best value: 0.551389:   4%|▎         | 7/200 [00:15<06:58,  2.17s/it]

[I 2025-11-24 17:20:14,195] Trial 6 finished with value: 0.5 and parameters: {'max_depth': 8, 'learning_rate': 0.25576937138930933, 'n_estimators': 592, 'subsample': 0.9797950839781646, 'colsample_bytree': 0.631486010085091, 'min_child_weight': 9, 'gamma': 3.4650828381598875, 'reg_alpha': 0.004458961318158366, 'reg_lambda': 0.013074447408771622, 'max_delta_step': 7}. Best is trial 4 with value: 0.5513888888888889.


KeyboardInterrupt: 